In [1]:
import requests
import time
import pandas as pd
from pandas import json_normalize
import os
from unidecode import unidecode
from itertools import product
import numpy as np
import re
import geopandas as gpd

#from unidecode import unidecode
#from geopy.geocoders import Nominatim
#from geopy.extra.rate_limiter import RateLimiter

In [2]:
!pip install unidecode

In [3]:
pip install geopandas shapely pyogrio

Note: you may need to restart the kernel to use updated packages.


In [4]:
# Load the API key from the .env file
from dotenv import load_dotenv


load_dotenv()                                          # Read the .env file
openchargemap_api_key = os.getenv("EV_API")               # Get the value of OPENCHARGEMAP_API_KEY
if openchargemap_api_key:
    print("Key loaded!")
else:
    print("ERROR: Key not found. Check your '.env' file.")

Key loaded!


## Extracting data from the API

In [5]:
# Step 1 - Endpoint (URL) - 
ev_url = "https://api.openchargemap.io/v3/poi"

#Step 2 -Importing the API key
headers = {
    "X-API-Key": openchargemap_api_key
}

#Step 3 - Defining the countries to download data
countries = ["ES","NL","DE","PT","FR"]

#Step 4 - creating an empty list to store data from all countries
all_data = []

# Step 5 - Looping through each country
for country in countries:

    params = {
        "output": "json",
        "countrycode": country,
        "maxresults": 30000
    }

    response = requests.get(
        ev_url,
        headers=headers,
        params=params
    )
    
#Step 6 - Checking if the request was successful
    if response.status_code == 200:

        data = response.json()

        print(f"{country}: {len(data)} stations downloaded")

        # Add country code to each record
        for station in data:
            station["CountryCode"] = country

        # Add data to master list
        all_data.extend(data)

        # Pause between requests
        time.sleep(1)

    else:
        print(f"Error downloading data for {country}")



ES: 20142 stations downloaded
NL: 8160 stations downloaded
DE: 24589 stations downloaded
PT: 3736 stations downloaded
FR: 16088 stations downloaded


## Normalizing and Analysing the data

In [6]:
raw_ev_df = pd.json_normalize(all_data)

In [7]:
raw_ev_df.sample()

,UserComments,PercentageSimilarity,MediaItems,IsRecentlyVerified,DateLastVerified,ID,UUID,ParentChargePointID,DataProviderID,DataProvidersReference,...,AddressInfo.Longitude,AddressInfo.ContactTelephone1,AddressInfo.ContactTelephone2,AddressInfo.ContactEmail,AddressInfo.AccessComments,AddressInfo.RelatedURL,AddressInfo.Distance,AddressInfo.DistanceUnit,UsageType,OperatorInfo
44101,None,None,None,False,2017-08-29T13:41:00Z,92176,D6E5E979-FFB9-4179-BB4F-A598233A2727,None,1,None,...,9.666451,None,None,None,"Nutzbar zu den Öffnungszeiten Mo-Fr 8-17, Sa 8...",http://www.autohaus-strnad.de/,None,0,NaN,NaN


In [8]:
#Check for duplicates
print("Total rows:", len(raw_ev_df))
print("Unique stations:", raw_ev_df["ID"].nunique())

Total rows: 72715
Unique stations: 72715


In [9]:
## Identifying the EV stations records were created and focussing from 2020 to 2024

In [10]:
# Convert DateCreated to datetime
raw_ev_df["DateCreated"] = pd.to_datetime(
    raw_ev_df["DateCreated"],
    errors="coerce"
)

# Extract the year
raw_ev_df["Station_Created_Year"] = raw_ev_df["DateCreated"].dt.year

# Countries to analyse
countries = ["ES", "NL", "DE", "PT", "FR"]

# Filter the dataframe
filtered_df = raw_ev_df[
    raw_ev_df["CountryCode"].isin(countries)
].copy()

# Count unique charging stations added per year for each country
stations_by_year_country = (
    filtered_df
    .groupby(["CountryCode", "Station_Created_Year"])["ID"]
    .nunique()
    .reset_index(name="Number_of_Stations_Added")
)

stations_by_year_country

,CountryCode,Station_Created_Year,Number_of_Stations_Added
0,DE,2011,13
1,DE,2012,8
2,DE,2013,46
3,DE,2014,53
4,DE,2015,4351
...,...,...,...
73,PT,2022,400
74,PT,2023,1148
75,PT,2024,18
76,PT,2025,415


In [11]:
stations_by_year_country_pivot = (
    stations_by_year_country
    .pivot(
        index="Station_Created_Year",
        columns="CountryCode",
        values="Number_of_Stations_Added"
    )
    .fillna(0)
    .astype(int)
)

stations_by_year_country_pivot

CountryCode,DE,ES,FR,NL,PT
Station_Created_Year,,,,,
2011,13,2,127,5,97
2012,8,0,41,26,82
2013,46,0,650,29,2
2014,53,79,125,58,134
2015,4351,64,41,5322,16
2016,2466,303,183,2051,13
2017,2897,208,440,81,76
2018,1996,226,227,48,104
2019,399,149,192,96,173


In [12]:
raw_ev_df.columns

Index(['UserComments', 'PercentageSimilarity', 'MediaItems',
       'IsRecentlyVerified', 'DateLastVerified', 'ID', 'UUID',
       'ParentChargePointID', 'DataProviderID', 'DataProvidersReference',
       'OperatorID', 'OperatorsReference', 'UsageTypeID', 'UsageCost',
       'Connections', 'NumberOfPoints', 'GeneralComments', 'DatePlanned',
       'DateLastConfirmed', 'StatusTypeID', 'DateLastStatusUpdate',
       'MetadataValues', 'DataQualityLevel', 'DateCreated',
       'SubmissionStatusTypeID', 'CountryCode', 'DataProvider.WebsiteURL',
       'DataProvider.Comments',
       'DataProvider.DataProviderStatusType.IsProviderEnabled',
       'DataProvider.DataProviderStatusType.ID',
       'DataProvider.DataProviderStatusType.Title',
       'DataProvider.IsRestrictedEdit', 'DataProvider.IsOpenDataLicensed',
       'DataProvider.IsApprovedImport', 'DataProvider.License',
       'DataProvider.DateLastImported', 'DataProvider.ID',
       'DataProvider.Title', 'OperatorInfo.WebsiteURL',
   

In [13]:
working_ev_df = raw_ev_df.copy()

In [14]:
working_ev_df.head()

,UserComments,PercentageSimilarity,MediaItems,IsRecentlyVerified,DateLastVerified,ID,UUID,ParentChargePointID,DataProviderID,DataProvidersReference,...,AddressInfo.ContactTelephone1,AddressInfo.ContactTelephone2,AddressInfo.ContactEmail,AddressInfo.AccessComments,AddressInfo.RelatedURL,AddressInfo.Distance,AddressInfo.DistanceUnit,UsageType,OperatorInfo,Station_Created_Year
0,None,None,None,True,2026-06-23T12:47:00Z,495071,3737F5E4-4AF8-40D6-A872-F460219AD236,None,1,None,...,None,None,None,None,None,None,0,NaN,NaN,2026
1,None,None,None,True,2026-06-23T12:41:00Z,495070,D5910A29-005C-43DE-A1A6-39338B9CC98F,None,1,None,...,None,None,None,None,None,None,0,NaN,NaN,2026
2,None,None,None,True,2026-06-23T12:37:00Z,495069,43505BA7-1535-41AD-8154-27F675A38DD6,None,1,None,...,None,None,None,None,None,None,0,NaN,NaN,2026
3,None,None,None,True,2026-06-23T12:32:00Z,495068,C903E57E-E0D1-42F8-8896-1A546226589C,None,1,None,...,None,None,None,None,None,None,0,NaN,NaN,2026
4,None,None,None,True,2026-06-23T12:25:00Z,495067,07D026C7-D363-483E-83BF-79B9DED47F66,None,1,None,...,None,None,None,None,None,None,0,NaN,NaN,2026


In [15]:
working_ev_df.isnull().sum()

UserComments                70921
PercentageSimilarity        72715
MediaItems                  62875
IsRecentlyVerified              0
DateLastVerified                0
                            ...  
AddressInfo.Distance        72715
AddressInfo.DistanceUnit        0
UsageType                   72715
OperatorInfo                72715
Station_Created_Year            0
Length: 86, dtype: int64

In [16]:
working_ev_df["CountryCode"].value_counts()

CountryCode
DE    24589
ES    20142
FR    16088
NL     8160
PT     3736
Name: count, dtype: int64

In [17]:
working_ev_df.columns

Index(['UserComments', 'PercentageSimilarity', 'MediaItems',
       'IsRecentlyVerified', 'DateLastVerified', 'ID', 'UUID',
       'ParentChargePointID', 'DataProviderID', 'DataProvidersReference',
       'OperatorID', 'OperatorsReference', 'UsageTypeID', 'UsageCost',
       'Connections', 'NumberOfPoints', 'GeneralComments', 'DatePlanned',
       'DateLastConfirmed', 'StatusTypeID', 'DateLastStatusUpdate',
       'MetadataValues', 'DataQualityLevel', 'DateCreated',
       'SubmissionStatusTypeID', 'CountryCode', 'DataProvider.WebsiteURL',
       'DataProvider.Comments',
       'DataProvider.DataProviderStatusType.IsProviderEnabled',
       'DataProvider.DataProviderStatusType.ID',
       'DataProvider.DataProviderStatusType.Title',
       'DataProvider.IsRestrictedEdit', 'DataProvider.IsOpenDataLicensed',
       'DataProvider.IsApprovedImport', 'DataProvider.License',
       'DataProvider.DateLastImported', 'DataProvider.ID',
       'DataProvider.Title', 'OperatorInfo.WebsiteURL',
   

In [18]:
working_ev_df["StatusType.IsOperational"].value_counts()

StatusType.IsOperational
True     68483
False     1575
Name: count, dtype: int64

In [19]:
working_ev_df["AddressInfo.Town"].value_counts().head(20)

AddressInfo.Town
Berlin        854
Hamburg       621
München       508
Madrid        451
Den Haag      434
Amsterdam     420
Rotterdam     349
              293
Barcelona     282
Stuttgart     275
Essen         254
Lisboa        230
Utrecht       213
Hannover      198
Paris         154
Düsseldorf    150
Bremen        139
Leipzig       136
Eindhoven     129
Köln          126
Name: count, dtype: int64

## Dropping unnecessary columns

In [20]:
working_ev_df.columns


Index(['UserComments', 'PercentageSimilarity', 'MediaItems',
       'IsRecentlyVerified', 'DateLastVerified', 'ID', 'UUID',
       'ParentChargePointID', 'DataProviderID', 'DataProvidersReference',
       'OperatorID', 'OperatorsReference', 'UsageTypeID', 'UsageCost',
       'Connections', 'NumberOfPoints', 'GeneralComments', 'DatePlanned',
       'DateLastConfirmed', 'StatusTypeID', 'DateLastStatusUpdate',
       'MetadataValues', 'DataQualityLevel', 'DateCreated',
       'SubmissionStatusTypeID', 'CountryCode', 'DataProvider.WebsiteURL',
       'DataProvider.Comments',
       'DataProvider.DataProviderStatusType.IsProviderEnabled',
       'DataProvider.DataProviderStatusType.ID',
       'DataProvider.DataProviderStatusType.Title',
       'DataProvider.IsRestrictedEdit', 'DataProvider.IsOpenDataLicensed',
       'DataProvider.IsApprovedImport', 'DataProvider.License',
       'DataProvider.DateLastImported', 'DataProvider.ID',
       'DataProvider.Title', 'OperatorInfo.WebsiteURL',
   

In [21]:
#Removing Columns that are not relevant to our analysis
cols_to_drop = ['UserComments', 'PercentageSimilarity', 'MediaItems','DateLastVerified', 'UUID','ParentChargePointID',
    'DataProviderID', 'DataProvidersReference', 'OperatorID', 'OperatorsReference',
    'GeneralComments', 'DatePlanned', 'DateLastConfirmed', 'DateLastStatusUpdate',
    'MetadataValues', 'DataQualityLevel',  'SubmissionStatusTypeID',
    'DataProvider.WebsiteURL', 'DataProvider.Comments',
    'DataProvider.DataProviderStatusType.IsProviderEnabled',
    'DataProvider.DataProviderStatusType.ID',
    'DataProvider.DataProviderStatusType.Title',
    'DataProvider.IsRestrictedEdit', 'DataProvider.IsOpenDataLicensed',
    'DataProvider.IsApprovedImport', 'DataProvider.License',
    'DataProvider.DateLastImported', 'DataProvider.ID',
    'DataProvider.Title', 'OperatorInfo.WebsiteURL',
    'OperatorInfo.Comments', 'OperatorInfo.PhonePrimaryContact',
    'OperatorInfo.PhoneSecondaryContact', 'OperatorInfo.IsPrivateIndividual',
    'OperatorInfo.AddressInfo', 'OperatorInfo.BookingURL',
    'OperatorInfo.ContactEmail', 'OperatorInfo.FaultReportEmail',
    'OperatorInfo.IsRestrictedEdit', 'OperatorInfo.ID',
    'OperatorInfo.Title', 'UsageType.IsPayAtLocation',
    'StatusType.IsUserSelectable', 'StatusType.ID',
    'SubmissionStatus.IsLive', 'SubmissionStatus.ID',
    'SubmissionStatus.Title', 'AddressInfo.ID',
    'AddressInfo.AddressLine2', 'AddressInfo.Postcode',
    'AddressInfo.CountryID', 'AddressInfo.Country.ISOCode',
    'AddressInfo.Country.ContinentCode', 'AddressInfo.Country.ID',
    'AddressInfo.ContactTelephone1', 'AddressInfo.ContactTelephone2',
    'AddressInfo.ContactEmail', 'AddressInfo.AccessComments',
    'AddressInfo.RelatedURL', 'AddressInfo.Distance',
    'AddressInfo.DistanceUnit'
]

working_ev_df = working_ev_df.drop(columns=cols_to_drop, errors='ignore')

working_ev_df.sample(10)

,IsRecentlyVerified,ID,UsageTypeID,UsageCost,Connections,NumberOfPoints,StatusTypeID,DateCreated,CountryCode,UsageType.IsMembershipRequired,...,AddressInfo.Title,AddressInfo.AddressLine1,AddressInfo.Town,AddressInfo.StateOrProvince,AddressInfo.Country.Title,AddressInfo.Latitude,AddressInfo.Longitude,UsageType,OperatorInfo,Station_Created_Year
11634,False,210650,4.0,"0,39€/kWh 22kW AC","[{'ID': 350967, 'ConnectionTypeID': 1036, 'Con...",2.0,50,2022-12-14 08:58:00+00:00,ES,True,...,Ayunt. El Ejido. Calle Bayarcal,Calle Bayarcal,El Ejido,Andalucía,Spain,36.773380,-2.807293,NaN,NaN,2022
58406,False,469301,1.0,None,"[{'ID': 787729, 'ConnectionTypeID': 33, 'Conne...",6.0,50,2025-11-12 19:14:00+00:00,FR,None,...,Carrefour Energies - Bayeux,7-9 Route de Vaux sur Aure,Bayeux,None,France,49.284411,-0.705488,NaN,NaN,2025
45014,False,86176,1.0,€1.73 + €0.3213/kWh; Andere Tarife möglich,"[{'ID': 122550, 'ConnectionTypeID': 25, 'Conne...",2.0,50,2017-04-28 06:06:00+00:00,DE,None,...,Friedrich Ebert Damm 120,Friedrich Ebert Damm 120,Wandsbek,Hamburg,Germany,53.587860,10.093890,NaN,NaN,2017
72256,False,20099,0.0,,[],NaN,50,2013-10-29 17:44:00+00:00,FR,None,...,Renault Retail Group Muret,254 Avenue des Pyrénées,,Muret,France,43.455220,1.395210,NaN,NaN,2013
60729,False,199007,1.0,None,"[{'ID': 332140, 'ConnectionTypeID': 28, 'Conne...",NaN,50,2022-08-23 10:36:00+00:00,FR,None,...,SIGEIF - 23 RUE CHANTE COQ - PUTEAUX,2 Rue Chante-Coq 92800 Puteaux,None,None,France,48.883921,2.236634,NaN,NaN,2022
44214,False,91631,4.0,None,"[{'ID': 129901, 'ConnectionTypeID': 25, 'Conne...",2.0,50,2017-08-09 11:22:00+00:00,DE,True,...,Hauptbahnhof Parkhaus P2,Heinrich-von-Stephan-Straße 6,Mannheim,None,Germany,49.478019,8.473005,NaN,NaN,2017
71528,False,77262,4.0,">3kW: 0,70€/5min ; Autres tarifs disponibles","[{'ID': 111164, 'ConnectionTypeID': 25, 'Conne...",3.0,50,2017-01-07 09:39:00+00:00,FR,True,...,Grenoble - Jean Pain - Stade des Alpes,Allée de lAncien Bastion,Grenoble,Isère,France,45.187972,5.738264,NaN,NaN,2017
36014,False,180554,NaN,None,"[{'ID': 283766, 'ConnectionTypeID': 25, 'Conne...",NaN,50,2021-07-08 10:00:00+00:00,DE,NaN,...,Kiefholzstraße 189,Kiefholzstraße 189,Berlin,None,Germany,52.465270,13.488120,NaN,NaN,2021
48188,False,61272,4.0,None,"[{'ID': 78002, 'ConnectionTypeID': 25, 'Connec...",2.0,50,2016-02-21 11:11:00+00:00,DE,True,...,Monheimer Elektrizitäts- und Gasversorgung (MEGA),Rheinpromenade 3a,Monheim,Kreis Mettmann,Germany,51.104114,6.888228,NaN,NaN,2016
29985,False,186800,NaN,None,"[{'ID': 300197, 'ConnectionTypeID': 33, 'Conne...",NaN,50,2021-07-12 02:49:00+00:00,DE,NaN,...,Buchholzer Str. 1,Buchholzer Str. 1,Teupitz,None,Germany,52.123445,13.638079,NaN,NaN,2021


In [22]:
print("DateCreated exists:", "DateCreated" in working_ev_df.columns)

[col for col in working_ev_df.columns if "Date" in col]

DateCreated exists: True


['DateCreated']

In [23]:
working_ev_df.columns

Index(['IsRecentlyVerified', 'ID', 'UsageTypeID', 'UsageCost', 'Connections',
       'NumberOfPoints', 'StatusTypeID', 'DateCreated', 'CountryCode',
       'UsageType.IsMembershipRequired', 'UsageType.IsAccessKeyRequired',
       'UsageType.ID', 'UsageType.Title', 'StatusType.IsOperational',
       'StatusType.Title', 'AddressInfo.Title', 'AddressInfo.AddressLine1',
       'AddressInfo.Town', 'AddressInfo.StateOrProvince',
       'AddressInfo.Country.Title', 'AddressInfo.Latitude',
       'AddressInfo.Longitude', 'UsageType', 'OperatorInfo',
       'Station_Created_Year'],
      dtype='object')

## Codes to arrive at Province and Autonomous Community

In [24]:
!pip install unidecode

In [25]:

# Dictionary for known duplicates / variants
mapping = {
    "comunidad de madrid": "Madrid",
    "community of madrid": "Madrid",
    "madrid ": "Madrid",
    "madrid.": "Madrid",

    "catalunya": "Catalonia",
    "catalonia": "Catalonia",

    "comunitat valenciana": "Valencian Community",
    "valencian community": "Valencian Community",

    "euskadi": "Basque Country",
    "autonomous community of the basque country": "Basque Country",

    "illes balears": "Balearic Islands",
    "isles baleares": "Balearic Islands",
    "balearic islands": "Balearic Islands",

    "castilla y leon": "Castile and León",
    "castile and leon": "Castile and León",

    "castilla-la mancha": "Castile-La Mancha",
    "castile-la mancha": "Castile-La Mancha",

    "region of murcia": "Murcia",
    "región de murcia": "Murcia",

    "galicia": "Galicia",
    "galicia ": "Galicia",

    "asturias / asturies": "Asturias",
    "asturias": "Asturias",

    "navarra / nafarroa": "Navarra",
    "navarre": "Navarra",

    "la rioja": "La Rioja",
    "rioja": "La Rioja",

    "canarias": "Canary Islands",
    "canary islands": "Canary Islands",
}

def clean_state(value):
    if pd.isna(value):
        return value

    # Convert to string
    v = str(value)

    # Remove extra spaces
    v = v.strip()

    # Normalize accents (Játiva → Jativa)
    v = unidecode(v)

    # Lowercase for matching
    v_low = v.lower()

    # Remove punctuation & double spaces
    v_low = re.sub(r"[^a-z0-9\s/]", "", v_low)
    v_low = re.sub(r"\s+", " ", v_low).strip()

    # Apply mapping if exists
    if v_low in mapping:
        return mapping[v_low]

    # Final formatting: Title Case
    return v.title()

In [26]:
working_ev_df["AddressInfo.StateOrProvince"] = working_ev_df["AddressInfo.StateOrProvince"].apply(clean_state)

In [27]:
working_ev_df["AddressInfo.StateOrProvince"].value_counts().head(20)

AddressInfo.StateOrProvince
Catalonia              2740
Valencian Community    1963
Madrid                 1879
Andalucia              1529
Castile and León       1062
Andalusia               823
Balearic Islands        818
                        759
Galicia                 754
Basque Country          564
Aragon                  500
Castilla-La Mancha      441
Hamburg                 421
Berlin                  399
Lisboa                  398
Cantabria               373
Canary Islands          355
Extremadura             347
Region De Murcia        327
Bayern                  311
Name: count, dtype: int64

In [28]:
#SPAIN ONLY
spain_working_ev_df = working_ev_df.loc[
    working_ev_df["CountryCode"] == "ES"
].copy()

spain_working_ev_df.head()

,IsRecentlyVerified,ID,UsageTypeID,UsageCost,Connections,NumberOfPoints,StatusTypeID,DateCreated,CountryCode,UsageType.IsMembershipRequired,...,AddressInfo.Title,AddressInfo.AddressLine1,AddressInfo.Town,AddressInfo.StateOrProvince,AddressInfo.Country.Title,AddressInfo.Latitude,AddressInfo.Longitude,UsageType,OperatorInfo,Station_Created_Year
0,True,495071,4.0,"0,54€/kWh DC","[{'ID': 842056, 'ConnectionTypeID': 33, 'Conne...",2.0,50,2026-06-23 12:45:00+00:00,ES,True,...,E.S. Q8 Gavá,Carrer de l'Enginy,Gavà,Catalonia,Spain,41.302207,2.016679,NaN,NaN,2026
1,True,495070,4.0,"0,60€/kWh","[{'ID': 842054, 'ConnectionTypeID': 33, 'Conne...",2.0,50,2026-06-23 12:40:00+00:00,ES,True,...,ALDI Torroella,GI-641,Torroella de Montgrí,Catalonia,Spain,42.041647,3.135979,NaN,NaN,2026
2,True,495069,4.0,"0,54€/kWh DC","[{'ID': 842053, 'ConnectionTypeID': 33, 'Conne...",2.0,50,2026-06-23 12:35:00+00:00,ES,True,...,E.S. Q8 Ripoll,Carrer dels Martinets,Ripoll,Catalonia,Spain,42.215653,2.168983,NaN,NaN,2026
3,True,495068,4.0,"0,37€/kWh","[{'ID': 842052, 'ConnectionTypeID': 33, 'Conne...",4.0,50,2026-06-23 12:30:00+00:00,ES,True,...,MapisaOil,Eix Salou - Andorra C-14,None,Catalonia,Spain,42.317620,1.388048,NaN,NaN,2026
4,True,495067,4.0,"0,25€/kWh","[{'ID': 842051, 'ConnectionTypeID': 25, 'Conne...",4.0,50,2026-06-23 12:24:00+00:00,ES,True,...,Mercadona Cervera,Carrer de Montoliu,Cervera,Catalonia,Spain,41.672195,1.269770,NaN,NaN,2026


In [29]:
# Create GeoDataFrame
gdf = gpd.GeoDataFrame(
    spain_working_ev_df.copy(),
    geometry=gpd.points_from_xy(
        spain_working_ev_df["AddressInfo.Longitude"],
        spain_working_ev_df["AddressInfo.Latitude"]
    ),
    crs="EPSG:4326"
)

In [30]:
# get spain comunity coordanates file and DF 
url = "https://gisco-services.ec.europa.eu/distribution/v2/nuts/geojson/NUTS_RG_01M_2024_4326_LEVL_2.geojson"

communities = gpd.read_file(url)

# Keep spain from the DF
communities = communities[communities["CNTR_CODE"] == "ES"].copy()

In [31]:
communities.sample(10)

,NUTS_ID,LEVL_CODE,CNTR_CODE,NAME_LATN,NUTS_NAME,MOUNT_TYPE,URBN_TYPE,COAST_TYPE,NAME_ENGL,NAME_FREN,ISO3_CODE,SVRG_UN,CAPT,EU_STAT,EFTA_STAT,CC_STAT,NAME_GERM,geometry
140,ES70,2,ES,Canarias,Canarias,NaN,None,None,Spain,Espagne,ESP,UN Member State,Madrid,T,F,F,Spanien,"MULTIPOLYGON (((-16.12763 28.57124, -16.12079 ..."
131,ES53,2,ES,Illes Balears,Illes Balears,NaN,None,None,Spain,Espagne,ESP,UN Member State,Madrid,T,F,F,Spanien,"MULTIPOLYGON (((3.17712 39.95349, 3.18081 39.9..."
133,ES13,2,ES,Cantabria,Cantabria,NaN,None,None,Spain,Espagne,ESP,UN Member State,Madrid,T,F,F,Spanien,"MULTIPOLYGON (((-3.58209 43.51122, -3.5785 43...."
122,ES24,2,ES,Aragón,Aragón,NaN,None,None,Spain,Espagne,ESP,UN Member State,Madrid,T,F,F,Spanien,"MULTIPOLYGON (((-0.75278 42.92407, -0.74109 42..."
132,ES12,2,ES,Principado de Asturias,Principado de Asturias,NaN,None,None,Spain,Espagne,ESP,UN Member State,Madrid,T,F,F,Spanien,"MULTIPOLYGON (((-5.83099 43.64236, -5.82892 43..."
129,ES11,2,ES,Galicia,Galicia,NaN,None,None,Spain,Espagne,ESP,UN Member State,Madrid,T,F,F,Spanien,"MULTIPOLYGON (((-7.69974 43.73511, -7.68584 43..."
130,ES41,2,ES,Castilla y León,Castilla y León,NaN,None,None,Spain,Espagne,ESP,UN Member State,Madrid,T,F,F,Spanien,"MULTIPOLYGON (((-4.85075 43.17325, -4.84444 43..."
139,ES21,2,ES,País Vasco,País Vasco,NaN,None,None,Spain,Espagne,ESP,UN Member State,Madrid,T,F,F,Spanien,"MULTIPOLYGON (((-2.7475 43.45203, -2.74657 43...."
134,ES61,2,ES,Andalucía,Andalucía,NaN,None,None,Spain,Espagne,ESP,UN Member State,Madrid,T,F,F,Spanien,"MULTIPOLYGON (((-5.00842 38.71525, -5.00102 38..."
142,ES22,2,ES,Comunidad Foral de Navarra,Comunidad Foral de Navarra,NaN,None,None,Spain,Espagne,ESP,UN Member State,Madrid,T,F,F,Spanien,"MULTIPOLYGON (((-1.66739 43.31443, -1.6527 43...."


In [32]:
joined = gpd.sjoin(
    gdf,
    communities,
    how="left",
    predicate="within"
)

In [33]:
joined.columns

Index(['IsRecentlyVerified', 'ID', 'UsageTypeID', 'UsageCost', 'Connections',
       'NumberOfPoints', 'StatusTypeID', 'DateCreated', 'CountryCode',
       'UsageType.IsMembershipRequired', 'UsageType.IsAccessKeyRequired',
       'UsageType.ID', 'UsageType.Title', 'StatusType.IsOperational',
       'StatusType.Title', 'AddressInfo.Title', 'AddressInfo.AddressLine1',
       'AddressInfo.Town', 'AddressInfo.StateOrProvince',
       'AddressInfo.Country.Title', 'AddressInfo.Latitude',
       'AddressInfo.Longitude', 'UsageType', 'OperatorInfo',
       'Station_Created_Year', 'geometry', 'index_right', 'NUTS_ID',
       'LEVL_CODE', 'CNTR_CODE', 'NAME_LATN', 'NUTS_NAME', 'MOUNT_TYPE',
       'URBN_TYPE', 'COAST_TYPE', 'NAME_ENGL', 'NAME_FREN', 'ISO3_CODE',
       'SVRG_UN', 'CAPT', 'EU_STAT', 'EFTA_STAT', 'CC_STAT', 'NAME_GERM'],
      dtype='object')

In [34]:
#Merge comunity Column
spain_working_ev_df["Autonomous_Community"] = joined["NAME_LATN"].values

In [35]:
spain_working_ev_df.head(20)

,IsRecentlyVerified,ID,UsageTypeID,UsageCost,Connections,NumberOfPoints,StatusTypeID,DateCreated,CountryCode,UsageType.IsMembershipRequired,...,AddressInfo.AddressLine1,AddressInfo.Town,AddressInfo.StateOrProvince,AddressInfo.Country.Title,AddressInfo.Latitude,AddressInfo.Longitude,UsageType,OperatorInfo,Station_Created_Year,Autonomous_Community
0,True,495071,4.0,"0,54€/kWh DC","[{'ID': 842056, 'ConnectionTypeID': 33, 'Conne...",2.0,50,2026-06-23 12:45:00+00:00,ES,True,...,Carrer de l'Enginy,Gavà,Catalonia,Spain,41.302207,2.016679,NaN,NaN,2026,Cataluña
1,True,495070,4.0,"0,60€/kWh","[{'ID': 842054, 'ConnectionTypeID': 33, 'Conne...",2.0,50,2026-06-23 12:40:00+00:00,ES,True,...,GI-641,Torroella de Montgrí,Catalonia,Spain,42.041647,3.135979,NaN,NaN,2026,Cataluña
2,True,495069,4.0,"0,54€/kWh DC","[{'ID': 842053, 'ConnectionTypeID': 33, 'Conne...",2.0,50,2026-06-23 12:35:00+00:00,ES,True,...,Carrer dels Martinets,Ripoll,Catalonia,Spain,42.215653,2.168983,NaN,NaN,2026,Cataluña
3,True,495068,4.0,"0,37€/kWh","[{'ID': 842052, 'ConnectionTypeID': 33, 'Conne...",4.0,50,2026-06-23 12:30:00+00:00,ES,True,...,Eix Salou - Andorra C-14,None,Catalonia,Spain,42.317620,1.388048,NaN,NaN,2026,Cataluña
4,True,495067,4.0,"0,25€/kWh","[{'ID': 842051, 'ConnectionTypeID': 25, 'Conne...",4.0,50,2026-06-23 12:24:00+00:00,ES,True,...,Carrer de Montoliu,Cervera,Catalonia,Spain,41.672195,1.269770,NaN,NaN,2026,Cataluña
5,True,495066,4.0,"0,55€/kWh","[{'ID': 842048, 'ConnectionTypeID': 33, 'Conne...",2.0,50,2026-06-23 12:14:00+00:00,ES,True,...,Carrer de la Serradora,None,Valencian Community,Spain,39.464418,-0.335692,NaN,NaN,2026,Comunitat Valenciana
6,True,495065,6.0,"0,35€/kWh","[{'ID': 842047, 'ConnectionTypeID': 25, 'Conne...",2.0,50,2026-06-23 12:13:00+00:00,ES,False,...,Avinguda de França,None,Valencian Community,Spain,39.458752,-0.343863,NaN,NaN,2026,Comunitat Valenciana
7,True,495064,6.0,"0,45€/kWh + parking fee","[{'ID': 842045, 'ConnectionTypeID': 25, 'Conne...",2.0,50,2026-06-23 11:46:00+00:00,ES,False,...,carrer Blanc,None,Balearic Islands,Spain,39.507904,2.532608,NaN,NaN,2026,Illes Balears
8,True,495063,4.0,"0,54€/kWh DC","[{'ID': 842044, 'ConnectionTypeID': 33, 'Conne...",2.0,50,2026-06-23 11:16:00+00:00,ES,True,...,Avenida de la Unión Europea,None,Canary Islands,Spain,27.765326,-15.558520,NaN,NaN,2026,Canarias
9,True,495062,4.0,"0,60€/kWh","[{'ID': 842042, 'ConnectionTypeID': 33, 'Conne...",2.0,50,2026-06-23 11:05:00+00:00,ES,True,...,Calle Marquesa de Pinares,None,Extremadura,Spain,38.920172,-6.344270,NaN,NaN,2026,Extremadura


In [36]:
spain_working_ev_df["Autonomous_Community"].value_counts()

Autonomous_Community
Cataluña                      3606
Andalucía                     2965
Comunitat Valenciana          2439
Comunidad de Madrid           2178
Castilla y León               1308
Illes Balears                  961
Galicia                        958
Canarias                       865
País Vasco                     784
Castilla-La Mancha             745
Aragón                         670
Principado de Asturias         535
Región de Murcia               516
Extremadura                    445
Comunidad Foral de Navarra     423
Cantabria                      413
La Rioja                       161
Ciudad de Melilla                7
Ciudad de Ceuta                  6
Name: count, dtype: int64

In [37]:
spain_working_ev_df["Autonomous_Community"].isna().sum()

np.int64(157)

In [38]:
#Lookm for "Autonomous_Community" that are NaN
missing_community = spain_working_ev_df[
    spain_working_ev_df["Autonomous_Community"].isna()
]

missing_community

,IsRecentlyVerified,ID,UsageTypeID,UsageCost,Connections,NumberOfPoints,StatusTypeID,DateCreated,CountryCode,UsageType.IsMembershipRequired,...,AddressInfo.AddressLine1,AddressInfo.Town,AddressInfo.StateOrProvince,AddressInfo.Country.Title,AddressInfo.Latitude,AddressInfo.Longitude,UsageType,OperatorInfo,Station_Created_Year,Autonomous_Community
22,True,494731,4.0,None,"[{'ID': 841200, 'ConnectionTypeID': 33, 'Conne...",3.0,50,2026-06-20 17:27:00+00:00,ES,True,...,"C/ Salvador Camacho, 9",Santa Eulària des Riu,Balearic Islands,Spain,38.984299,1.543400,NaN,NaN,2026,NaN
420,True,480400,4.0,"0,60€/kWh","[{'ID': 806380, 'ConnectionTypeID': 33, 'Conne...",4.0,50,2026-03-16 06:55:00+00:00,ES,True,...,Paseo marítimo de Bouzas,None,Galicia,Spain,42.227176,-8.753985,NaN,NaN,2026,NaN
422,True,480244,4.0,,"[{'ID': 806173, 'ConnectionTypeID': 25, 'Conne...",2.0,50,2026-03-13 19:54:00+00:00,ES,True,...,Passeig de Rafael Campalans,Torredembarra,Catalonia,Spain,41.133696,1.399976,NaN,NaN,2026,NaN
443,True,479752,4.0,"0,45€/kWh","[{'ID': 805481, 'ConnectionTypeID': 33, 'Conne...",6.0,50,2026-03-04 23:02:00+00:00,ES,True,...,carrer Antoni Maria Alcover,None,Balearic Islands,Spain,39.529885,2.565432,NaN,NaN,2026,NaN
525,True,478125,5.0,"0,40€/kWh + parking fee","[{'ID': 802880, 'ConnectionTypeID': 25, 'Conne...",2.0,50,2026-02-14 15:04:00+00:00,ES,False,...,Moll de Barcelona,None,Catalonia,Spain,41.371660,2.181437,NaN,NaN,2026,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19768,False,70958,7.0,Free,"[{'ID': 101670, 'ConnectionTypeID': 25, 'Conne...",1.0,50,2016-08-21 18:55:00+00:00,ES,False,...,Muelle de Cruceros,Puerto del Rosario,Las Palmas,Spain,28.495497,-13.859205,NaN,NaN,2016,NaN
19823,False,61116,6.0,None,"[{'ID': 77723, 'ConnectionTypeID': 25, 'Connec...",3.0,50,2016-02-13 00:24:00+00:00,ES,False,...,Travesía de Bouzas Portuario,Vigo,Vigo,Spain,42.230199,-8.747678,NaN,NaN,2016,NaN
19825,False,61114,1.0,None,"[{'ID': 77716, 'ConnectionTypeID': 25, 'Connec...",NaN,0,2016-02-13 00:24:00+00:00,ES,None,...,"A-8, 22",Bilbao,Bilbao,Spain,42.208634,24.043651,NaN,NaN,2016,NaN
19853,False,61068,1.0,None,"[{'ID': 77618, 'ConnectionTypeID': 16, 'Connec...",NaN,50,2016-02-13 00:23:00+00:00,ES,None,...,"Via Roma, 4",Balears,Balearic Islands,Spain,40.579369,17.036496,NaN,NaN,2016,NaN


In [39]:
len(missing_community)

157

In [40]:
#Replace "Autonomous_Community" Nan with "AddressInfo.StateOrProvince" Column with values
mask = (
    spain_working_ev_df["Autonomous_Community"].isna() &
    spain_working_ev_df["AddressInfo.StateOrProvince"].notna()
)

spain_working_ev_df.loc[mask, "Autonomous_Community"] = (
    spain_working_ev_df.loc[mask, "AddressInfo.StateOrProvince"]
)

In [41]:
spain_working_ev_df["Autonomous_Community"].isna().sum()

np.int64(40)

In [42]:
#Look for towns to search de Community with AI
missing_towns = sorted(
    spain_working_ev_df.loc[
        spain_working_ev_df["Autonomous_Community"].isna(),
        "AddressInfo.Town"
    ].dropna().unique()
)

print(missing_towns)

['Alicante', 'Alporchinos', 'Arrecife', 'Barcelona', 'Barcelona ', 'Candelaria', 'Chaves', 'Coruña', 'Gibraltar', 'Gomera', 'Lagoa', 'Los Cristianos', 'Marbella', 'Mataró', 'Mazarrón', 'Morro Jable', 'O Grove', 'Olhao', 'Roses', 'Santa Cruz de la Palma', 'Santa Pola', 'Santiago del Teide', 'Soto Del Real ', 'Tarragona', 'Vilagarcia']


In [43]:
def fill_missing_community(town):

    if pd.isna(town):
        return None

    town = str(town).strip().lower()

    mapping = {
        # Comunidad Valenciana
        "alicante": "valencian community",
        "santa pola": "valencian community",

        # Cataluña
        "barcelona": "catalonia",
        "mataro": "catalonia",
        "tarragona": "catalonia",
        "roses": "catalonia",

        # Andalucía
        "marbella": "andalusia",
        "alporchinos": "andalusia",

        # Región de Murcia
        "mazarron": "region of murcia",

        # Canarias
        "arrecife": "canary islands",
        "candelaria": "canary islands",
        "gomera": "canary islands",
        "los cristianos": "canary islands",
        "morro jable": "canary islands",
        "santa cruz de la palma": "canary islands",
        "santiago del teide": "canary islands",

        # Galicia
        "coruna": "galicia",
        "o grove": "galicia",
        "vilagarcia": "galicia",

        # Comunidad de Madrid
        "soto del real": "community of madrid",

        # Out of spain
        "chaves": None,
        "gibraltar": None,
        "lagoa": None,
        "olhao": None
    }

    return mapping.get(town, None)

In [44]:
#Replaces values with info gathere4d from the town dictionary
mask = spain_working_ev_df["Autonomous_Community"].isna()

spain_working_ev_df.loc[mask, "Autonomous_Community"] = (
    spain_working_ev_df.loc[mask, "AddressInfo.Town"]
    .apply(fill_missing_community)
)

In [45]:
#Check fot unuique values and noralize
spain_working_ev_df["Autonomous_Community"].unique()

array(['Cataluña', 'Comunitat Valenciana ', 'Illes Balears', 'Canarias',
       'Extremadura', 'Comunidad de Madrid', 'Aragón',
       'Comunidad Foral de Navarra', 'Castilla y León', 'País Vasco',
       'Andalucía', 'Balearic Islands', 'Castilla-La Mancha', 'Galicia',
       'Principado de Asturias', 'Región de Murcia', 'La Rioja',
       'Cantabria', 'Catalonia', 'andalusia', 'Andalucia',
       'valencian community', 'Canary Islands', 'canary islands',
       'catalonia', 'galicia', None, 'Alicante', 'community of madrid',
       'Region De Murcia', 'Ciudad de Melilla', 'Ciudad de Ceuta',
       'Valencian Community', 'Basque Country', 'Murcia', 'Asturias',
       'Andalusia', 'Marbella', 'Roses', 'Barcelona', 'Tarragona',
       'Las Palmas', 'Vigo', 'Bilbao'], dtype=object)

In [46]:
community_mapping = {
    # Andalucía
    "Andalucía": "Andalusia",
    "Andalucia": "Andalusia",
    "andalusia": "Andalusia",
    "Andalusia": "Andalusia",
    "Marbella": "Andalusia",

    # Aragón
    "Aragón": "Aragon",

    # Asturias
    "Principado de Asturias": "Asturias",
    "Asturias": "Asturias",

    # Islas Baleares
    "Illes Balears": "Balearic Islands",
    "Balearic Islands": "Balearic Islands",

    # Canarias
    "Canarias": "Canary Islands",
    "Canary Islands": "Canary Islands",
    "canary islands": "Canary Islands",
    "Las Palmas": "Canary Islands",

    # Cantabria
    "Cantabria": "Cantabria",

    # Castilla-La Mancha
    "Castilla-La Mancha": "Castile-La Mancha",

    # Castilla y León
    "Castilla y León": "Castile and León",

    # Cataluña
    "Cataluña": "Catalonia",
    "Catalonia": "Catalonia",
    "catalonia": "Catalonia",
    "Barcelona": "Catalonia",
    "Tarragona": "Catalonia",
    "Roses": "Catalonia",

    # Comunidad Valenciana
    "Comunitat Valenciana": "Valencian Community",
    "Comunitat Valenciana ": "Valencian Community",
    "Valencian Community": "Valencian Community",
    "valencian community": "Valencian Community",
    "Alicante": "Valencian Community",

    # Madrid
    "Comunidad de Madrid": "Community of Madrid",
    "community of madrid": "Community of Madrid",

    # Extremadura
    "Extremadura": "Extremadura",

    # Galicia
    "Galicia": "Galicia",
    "galicia": "Galicia",
    "Vigo": "Galicia",

    # Murcia
    "Región de Murcia": "Region of Murcia",
    "Region De Murcia": "Region of Murcia",
    "Murcia": "Region of Murcia",

    # Navarra
    "Comunidad Foral de Navarra": "Navarre",

    # País Vasco
    "País Vasco": "Basque Country",
    "Basque Country": "Basque Country",
    "Bilbao": "Basque Country",

    # La Rioja
    "La Rioja": "La Rioja",

    # Ceuta y Melilla
    "Ciudad de Ceuta": "Ceuta",
    "Ciudad de Melilla": "Melilla"
}

In [47]:
spain_working_ev_df["Autonomous_Community"] = (
    spain_working_ev_df["Autonomous_Community"]
    .str.strip()
    .replace(community_mapping)
)

In [48]:
sorted(spain_working_ev_df["Autonomous_Community"].dropna().unique())

['Andalusia',
 'Aragon',
 'Asturias',
 'Balearic Islands',
 'Basque Country',
 'Canary Islands',
 'Cantabria',
 'Castile and León',
 'Castile-La Mancha',
 'Catalonia',
 'Ceuta',
 'Community of Madrid',
 'Extremadura',
 'Galicia',
 'La Rioja',
 'Melilla',
 'Navarre',
 'Region of Murcia',
 'Valencian Community']

In [49]:
#Look the number of autonomous_community (Total 17 + Melilla and Ceuta)
len(spain_working_ev_df["Autonomous_Community"].unique())

20

In [50]:
#Look for the Final Nan to drop or replace
stations_by_region = (
    spain_working_ev_df.groupby("Autonomous_Community")["ID"]
    .nunique()
    .sort_values(ascending=False)
)
stations_by_region

Autonomous_Community
Catalonia              3641
Andalusia              2980
Valencian Community    2457
Community of Madrid    2179
Castile and León       1308
Balearic Islands        987
Galicia                 967
Canary Islands          886
Basque Country          789
Castile-La Mancha       745
Aragon                  670
Asturias                536
Region of Murcia        520
Extremadura             445
Navarre                 423
Cantabria               418
La Rioja                161
Melilla                   7
Ceuta                     6
Name: ID, dtype: int64

In [51]:
spain_working_ev_df[spain_working_ev_df["Autonomous_Community"].isna()]

,IsRecentlyVerified,ID,UsageTypeID,UsageCost,Connections,NumberOfPoints,StatusTypeID,DateCreated,CountryCode,UsageType.IsMembershipRequired,...,AddressInfo.AddressLine1,AddressInfo.Town,AddressInfo.StateOrProvince,AddressInfo.Country.Title,AddressInfo.Latitude,AddressInfo.Longitude,UsageType,OperatorInfo,Station_Created_Year,Autonomous_Community
1970,False,469393,6.0,None,"[{'ID': 787934, 'ConnectionTypeID': 25, 'Conne...",1.0,50,2025-11-14 16:33:00+00:00,ES,False,...,Mataró,Mataró,None,Spain,41.530630,2.443868,NaN,NaN,2025,None
2736,False,460663,1.0,"0,38€/kwh","[{'ID': 775025, 'ConnectionTypeID': 2, 'Connec...",1.0,50,2025-09-01 13:01:00+00:00,ES,None,...,Avenida Vallesequillo,None,None,Spain,1.000000,1.000000,NaN,NaN,2025,None
2938,False,459435,4.0,"0,60€/kWh","[{'ID': 772559, 'ConnectionTypeID': 33, 'Conne...",2.0,50,2025-07-11 07:56:00+00:00,ES,True,...,Olhao,Olhao,None,Spain,37.124759,-7.770603,NaN,NaN,2025,None
2940,False,459433,4.0,"0,60€/kWh","[{'ID': 772557, 'ConnectionTypeID': 33, 'Conne...",2.0,50,2025-07-11 07:51:00+00:00,ES,True,...,Lagoa,Lagoa,None,Spain,37.151691,-8.426231,NaN,NaN,2025,None
2941,False,459432,4.0,"0,60€/kWh","[{'ID': 772556, 'ConnectionTypeID': 33, 'Conne...",2.0,50,2025-07-11 07:47:00+00:00,ES,True,...,Chaves,Chaves,None,Spain,41.731781,-7.469077,NaN,NaN,2025,None
2944,False,459280,1.0,price in place to plug,"[{'ID': 772321, 'ConnectionTypeID': 25, 'Conne...",1.0,50,2025-07-04 14:03:00+00:00,ES,None,...,Passeig Maritim Sant Joan De Deu,None,None,Spain,41.189072,1.605256,NaN,NaN,2025,None
2945,False,459279,1.0,price in place to plug,"[{'ID': 772320, 'ConnectionTypeID': 25, 'Conne...",1.0,50,2025-07-04 14:00:00+00:00,ES,None,...,Passeig Maritim De Sant Joan De Deu,None,None,Spain,41.186085,1.606471,NaN,NaN,2025,None
3233,False,312536,4.0,"0,50€/kWh AC","[{'ID': 602190, 'ConnectionTypeID': 25, 'Conne...",2.0,50,2025-05-10 20:40:00+00:00,ES,True,...,gibraltar,Gibraltar,None,Spain,36.116414,-5.349911,NaN,NaN,2025,None
3921,False,307255,4.0,"0,43€/kWh DC - 0,38€/kWh AC","[{'ID': 586182, 'ConnectionTypeID': 33, 'Conne...",5.0,50,2024-12-19 19:43:00+00:00,ES,True,...,Marina Coruña,Coruña,None,Spain,43.369589,-8.388005,NaN,NaN,2024,None
4842,False,303274,4.0,,"[{'ID': 570567, 'ConnectionTypeID': 33, 'Conne...",2.0,50,2024-08-21 10:16:00+00:00,ES,True,...,"Pg. de Joan de Borbó, 103",None,None,Spain,41.364809,2.186176,NaN,NaN,2024,None


In [52]:
towns_missing = (
    spain_working_ev_df.loc[
        spain_working_ev_df["Autonomous_Community"].isna(),
        "AddressInfo.Town"
    ]
    .dropna()
    .unique()
    .tolist()
)

print(towns_missing)

['Mataró', 'Olhao', 'Lagoa', 'Chaves', 'Gibraltar', 'Coruña', 'Mazarrón']


In [53]:
towns_to_drop = ["Olhao", "Lagoa", "Chaves", "Gibraltar"]

spain_working_ev_df = spain_working_ev_df[
    ~spain_working_ev_df["AddressInfo.Town"].isin(towns_to_drop)
].copy()

In [54]:
community_mapping = {
    "Mataró": "Catalonia",
    "Coruña": "Galicia",
    "Mazarrón": "Region of Murcia"
}

mask = spain_working_ev_df["Autonomous_Community"].isna()

spain_working_ev_df.loc[mask, "Autonomous_Community"] = (
    spain_working_ev_df.loc[mask, "AddressInfo.Town"]
    .map(community_mapping)
)

In [55]:
spain_working_ev_df = spain_working_ev_df.dropna(subset=["Autonomous_Community"])

In [56]:
spain_working_ev_df["Autonomous_Community"].isna().sum()

np.int64(0)

In [57]:
spain_working_ev_df["Autonomous_Community"].value_counts()

Autonomous_Community
Catalonia              3642
Andalusia              2980
Valencian Community    2457
Community of Madrid    2179
Castile and León       1308
Balearic Islands        987
Galicia                 968
Canary Islands          886
Basque Country          789
Castile-La Mancha       745
Aragon                  670
Asturias                536
Region of Murcia        521
Extremadura             445
Navarre                 423
Cantabria               418
La Rioja                161
Melilla                   7
Ceuta                     6
Name: count, dtype: int64

## Looking at the Highway perspective

In [58]:
# Make a copy
ev_df = spain_working_ev_df.copy()

# Convert DateCreated to datetime
ev_df["DateCreated"] = pd.to_datetime(ev_df["DateCreated"], errors="coerce")
ev_df["Station_Created_Year"] = ev_df["DateCreated"].dt.year

# Keep only 2020 to 2024
ev_2020_2024_df = ev_df[
    ev_df["Station_Created_Year"].between(2020, 2024)
].copy()

print("Rows from 2020-2024:", len(ev_2020_2024_df))

Rows from 2020-2024: 15278


In [59]:
ev_2020_2024_df["France_Spain_Corridor"] = (
    ev_2020_2024_df["AddressInfo.Latitude"].between(40.5, 42.9) &
    ev_2020_2024_df["AddressInfo.Longitude"].between(0.5, 3.5)
)

print("France-Spain corridor stations:",
      ev_2020_2024_df["France_Spain_Corridor"].sum())

France-Spain corridor stations: 2627


In [60]:
total_stations = len(ev_2020_2024_df)

corridor_stations = ev_2020_2024_df["France_Spain_Corridor"].sum()

percentage = corridor_stations / total_stations * 100

print(f"{percentage:.1f}%")

17.2%


In [61]:
ev_2020_2024_df["Portugal_Spain_Corridor"] = (
    ev_2020_2024_df["AddressInfo.Latitude"].between(37.0, 42.3) &
    ev_2020_2024_df["AddressInfo.Longitude"].between(-9.5, -5.5)
)

print("Portugal-Spain corridor stations:",
      ev_2020_2024_df["Portugal_Spain_Corridor"].sum())

Portugal-Spain corridor stations: 1347


In [62]:
ev_2020_2024_df["Mediterranean_Corridor"] = (
    ev_2020_2024_df["AddressInfo.Latitude"].between(37.5, 42.5) &
    ev_2020_2024_df["AddressInfo.Longitude"].between(-1.0, 3.5)
)

print("Mediterranean corridor stations:",
      ev_2020_2024_df["Mediterranean_Corridor"].sum())

Mediterranean corridor stations: 5421


In [63]:
ev_2020_2024_df["Madrid_Hub"] = (
    ev_2020_2024_df["AddressInfo.Latitude"].between(39.8, 41.2) &
    ev_2020_2024_df["AddressInfo.Longitude"].between(-4.5, -2.8)
)

print("Madrid hub stations:",
      ev_2020_2024_df["Madrid_Hub"].sum())

Madrid hub stations: 1954


In [64]:
ev_2020_2024_df["Madrid_Valencia_Corridor"] = (
    ev_2020_2024_df["AddressInfo.Latitude"].between(38.8, 40.8) &
    ev_2020_2024_df["AddressInfo.Longitude"].between(-4.2, -0.2)
)

print("Madrid-Valencia corridor stations:",
      ev_2020_2024_df["Madrid_Valencia_Corridor"].sum())

Madrid-Valencia corridor stations: 2975


In [65]:
ev_2020_2024_df["Madrid_Andalusia_Corridor"] = (
    ev_2020_2024_df["AddressInfo.Latitude"].between(36.3, 40.6) &
    ev_2020_2024_df["AddressInfo.Longitude"].between(-5.5, -2.8)
)

print("Madrid-Andalusia corridor stations:",
      ev_2020_2024_df["Madrid_Andalusia_Corridor"].sum())

Madrid-Andalusia corridor stations: 3117


In [66]:
ev_2020_2024_df["Tourism_Barcelona_Catalonia"] = (
    ev_2020_2024_df["AddressInfo.Latitude"].between(41.0, 42.5) &
    ev_2020_2024_df["AddressInfo.Longitude"].between(1.5, 3.3)
)

ev_2020_2024_df["Tourism_Madrid"] = (
    ev_2020_2024_df["AddressInfo.Latitude"].between(39.8, 41.2) &
    ev_2020_2024_df["AddressInfo.Longitude"].between(-4.5, -2.8)
)

ev_2020_2024_df["Tourism_Andalusia"] = (
    ev_2020_2024_df["AddressInfo.Latitude"].between(36.0, 38.8) &
    ev_2020_2024_df["AddressInfo.Longitude"].between(-7.5, -1.5)
)

ev_2020_2024_df["Tourism_Valencia_Alicante"] = (
    ev_2020_2024_df["AddressInfo.Latitude"].between(37.8, 40.2) &
    ev_2020_2024_df["AddressInfo.Longitude"].between(-1.2, 0.8)
)

ev_2020_2024_df["Tourism_Balearic_Islands"] = (
    ev_2020_2024_df["AddressInfo.Latitude"].between(38.5, 40.2) &
    ev_2020_2024_df["AddressInfo.Longitude"].between(1.0, 4.5)
)

In [67]:
angle_columns = [
    "France_Spain_Corridor",
    "Portugal_Spain_Corridor",
    "Mediterranean_Corridor",
    "Madrid_Hub",
    "Madrid_Valencia_Corridor",
    "Madrid_Andalusia_Corridor",
    "Tourism_Barcelona_Catalonia",
    "Tourism_Madrid",
    "Tourism_Andalusia",
    "Tourism_Valencia_Alicante",
    "Tourism_Balearic_Islands"
]

summary = pd.DataFrame({
    "Angle": angle_columns,
    "Number_of_Stations": [ev_2020_2024_df[col].sum() for col in angle_columns]
})

summary = summary.sort_values("Number_of_Stations", ascending=False)

summary

,Angle,Number_of_Stations
2,Mediterranean_Corridor,5421
5,Madrid_Andalusia_Corridor,3117
4,Madrid_Valencia_Corridor,2975
0,France_Spain_Corridor,2627
8,Tourism_Andalusia,2506
6,Tourism_Barcelona_Catalonia,2061
9,Tourism_Valencia_Alicante,2039
3,Madrid_Hub,1954
7,Tourism_Madrid,1954
1,Portugal_Spain_Corridor,1347


In [68]:
ev_2020_2024_df.to_csv("ev_spain_2020_2024_corridor_analysis2.csv", index=False)

print("File saved successfully!")

File saved successfully!


## Extracting to excel

## EUROSTAT API


In [69]:
url = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/road_eqs_carpda?geo=ES"

response = requests.get(url)
response.raise_for_status()
data = response.json()



def jsonstat_to_df(data):
    # Get the dimension names from the Eurostat data
    # Example: ["freq", "unit", "mot_nrg", "geo", "time"]
    dim_ids = data["id"]

    # This contains the category information for each dimension
    dimensions = data["dimension"]

    # These are the actual numbers from Eurostat
    values = data["value"]

    # This list will store all the possible codes for each dimension
    all_codes = []

    # Go through each dimension one by one
    for dim in dim_ids:
        category = dimensions[dim]["category"]

        # The index tells us the order of the codes
        index = category["index"]

        # Sometimes Eurostat gives the index as a dictionary
        # Example: {"ELC": 0, "PET": 1}
        if isinstance(index, dict):
            codes = sorted(index, key=index.get)

        # Sometimes Eurostat gives the index as a list
        # Example: ["ELC", "PET"]
        else:
            codes = index

        # Save the ordered codes for this dimension
        all_codes.append(codes)

    # This will become the final table
    rows = []

    # product(*all_codes) creates every possible combination
    # Example:
    # geo = ["ES"]
    # time = ["2023", "2024"]
    # mot_nrg = ["PET", "ELC"]
    #
    # It creates:
    # ES, 2023, PET
    # ES, 2023, ELC
    # ES, 2024, PET
    # ES, 2024, ELC
    for flat_index, codes_combo in enumerate(product(*all_codes)):

        # Eurostat can store values as a dictionary or as a list.
        # This handles both cases.
        if isinstance(values, dict):
            value = values.get(str(flat_index))
        else:
            value = values[flat_index]

        # Skip empty values
        if value is None:
            continue

        # This dictionary will become one row in the DataFrame
        row = {}

        # Match each dimension name with its code
        # Example:
        # dim = "mot_nrg"
        # code = "ELC"
        for dim, code in zip(dim_ids, codes_combo):

            # Get the readable label if Eurostat provides one
            # Example: "ELC" becomes "Electricity"
            label = dimensions[dim]["category"].get("label", {}).get(code, code)

            # Add the raw code column
            row[dim] = code

            # Add the readable label column
            row[f"{dim}_label"] = label

        # Add the actual Eurostat number
        row["value"] = value

        # Save this row
        rows.append(row)

    # Convert the list of rows into a pandas table
    return pd.DataFrame(rows)

    # Convert all rows into a pandas DataFrame
    return pd.DataFrame(rows)

df = jsonstat_to_df(data)

df.head()


,freq,freq_label,unit,unit_label,mot_nrg,mot_nrg_label,geo,geo_label,time,time_label,value
0,A,Annual,NR,Number,TOTAL,Total,ES,Spain,2013,2013,22025000
1,A,Annual,NR,Number,TOTAL,Total,ES,Spain,2014,2014,22029512
2,A,Annual,NR,Number,TOTAL,Total,ES,Spain,2015,2015,22355549
3,A,Annual,NR,Number,TOTAL,Total,ES,Spain,2016,2016,22876830
4,A,Annual,NR,Number,TOTAL,Total,ES,Spain,2017,2017,24676557


In [70]:
df["mot_nrg_label"].unique()

array(['Total', 'Petrol', 'Liquefied petroleum gases (LPG)', 'Diesel',
       'Natural gas', 'Electricity', 'Alternative energy',
       'Petrol (excluding hybrids) \xa0', 'Hybrid electric-petrol',
       'Plug-in hybrid petrol-electric \xa0',
       'Diesel (excluding hybrids) \xa0', 'Hybrid diesel-electric',
       'Plug-in hybrid diesel-electric \xa0',
       'Hydrogen and fuel cells\xa0', 'Bioethanol', 'Biodiesel',
       'Bi-fuel', 'Other'], dtype=object)

In [71]:
rows_to_drop = df[
    df["mot_nrg_label"].isin(["Diesel", "Petrol"])
].index

df = df.drop(rows_to_drop)

df

,freq,freq_label,unit,unit_label,mot_nrg,mot_nrg_label,geo,geo_label,time,time_label,value
0,A,Annual,NR,Number,TOTAL,Total,ES,Spain,2013,2013,22025000
1,A,Annual,NR,Number,TOTAL,Total,ES,Spain,2014,2014,22029512
2,A,Annual,NR,Number,TOTAL,Total,ES,Spain,2015,2015,22355549
3,A,Annual,NR,Number,TOTAL,Total,ES,Spain,2016,2016,22876830
4,A,Annual,NR,Number,TOTAL,Total,ES,Spain,2017,2017,24676557
...,...,...,...,...,...,...,...,...,...,...,...
192,A,Annual,NR,Number,OTH,Other,ES,Spain,2020,2020,10814
193,A,Annual,NR,Number,OTH,Other,ES,Spain,2021,2021,11578
194,A,Annual,NR,Number,OTH,Other,ES,Spain,2022,2022,11420
195,A,Annual,NR,Number,OTH,Other,ES,Spain,2023,2023,11406


In [72]:
#we just want to focuson EV vehicles so we will filter the data to only include rows where the "mot_nrg_label" column contains the word "electric"

electric_spain = df[df["mot_nrg_label"].str.contains("electric", case=False, na=False)]

In [73]:
#looking just at 2024 data in spain 
electric_spain_2024 = electric_spain[electric_spain["time"] == "2024"]

In [74]:
url = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/road_eqs_carpda?geo=PT&lang=en"

response = requests.get(url)
data_portugal = response.json()

df_portugal = jsonstat_to_df(data_portugal)

In [75]:
url = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/road_eqs_carpda?geo=FR&lang=en"

response = requests.get(url)
data_france = response.json()

df_france = jsonstat_to_df(data_france)

In [76]:
url = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/road_eqs_carpda?geo=NL&lang=en"

response = requests.get(url)
data_netherlands = response.json()

df_netherlands = jsonstat_to_df(data_netherlands)

In [77]:
url = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/road_eqs_carpda?geo=DE&lang=en"

response = requests.get(url)
data_germany = response.json()

df_germany = jsonstat_to_df(data_germany)

In [78]:
url = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/road_eqs_carhab?geo=PT&lang=en"

response = requests.get(url)
data_carhab_pt = response.json()

df_carhab_pt = jsonstat_to_df(data_carhab_pt)

df_carhab_pt.head()

,freq,freq_label,unit,unit_label,geo,geo_label,time,time_label,value
0,A,Annual,NR,Number,PT,Portugal,1990,1990,256
1,A,Annual,NR,Number,PT,Portugal,1991,1991,279
2,A,Annual,NR,Number,PT,Portugal,1992,1992,306
3,A,Annual,NR,Number,PT,Portugal,1993,1993,330
4,A,Annual,NR,Number,PT,Portugal,1994,1994,353


In [79]:
#merging the dataframes together to have a complete dataset for all the countries   
df_total = pd.concat([df, df_portugal, df_france, df_netherlands, df_germany], ignore_index=True)
df_total

,freq,freq_label,unit,unit_label,mot_nrg,mot_nrg_label,geo,geo_label,time,time_label,value
0,A,Annual,NR,Number,TOTAL,Total,ES,Spain,2013,2013,22025000
1,A,Annual,NR,Number,TOTAL,Total,ES,Spain,2014,2014,22029512
2,A,Annual,NR,Number,TOTAL,Total,ES,Spain,2015,2015,22355549
3,A,Annual,NR,Number,TOTAL,Total,ES,Spain,2016,2016,22876830
4,A,Annual,NR,Number,TOTAL,Total,ES,Spain,2017,2017,24676557
...,...,...,...,...,...,...,...,...,...,...,...
948,A,Annual,NR,Number,OTH,Other,DE,Germany,2020,2020,9514
949,A,Annual,NR,Number,OTH,Other,DE,Germany,2021,2021,9190
950,A,Annual,NR,Number,OTH,Other,DE,Germany,2022,2022,8812
951,A,Annual,NR,Number,OTH,Other,DE,Germany,2023,2023,8394


In [80]:
df_total["geo"].value_counts()


geo
FR    216
PT    214
NL    192
ES    175
DE    156
Name: count, dtype: int64

In [81]:
df_total.isna().sum()


freq             0
freq_label       0
unit             0
unit_label       0
mot_nrg          0
mot_nrg_label    0
geo              0
geo_label        0
time             0
time_label       0
value            0
dtype: int64

In [82]:
df_total["time"] = df_total["time"].astype(int)
df_total.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 953 entries, 0 to 952
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   freq           953 non-null    object
 1   freq_label     953 non-null    object
 2   unit           953 non-null    object
 3   unit_label     953 non-null    object
 4   mot_nrg        953 non-null    object
 5   mot_nrg_label  953 non-null    object
 6   geo            953 non-null    object
 7   geo_label      953 non-null    object
 8   time           953 non-null    int64 
 9   time_label     953 non-null    object
 10  value          953 non-null    int64 
dtypes: int64(2), object(9)
memory usage: 82.0+ KB


In [83]:
df_total.duplicated().sum()

np.int64(0)

In [84]:
#filtering the data to only include rows where the "mot_nrg_label" column contains the word "electric"
df_total_ev= df_total[df_total["mot_nrg_label"].str.contains("electric", case=False, na=False)]
df_total_ev 

,freq,freq_label,unit,unit_label,mot_nrg,mot_nrg_label,geo,geo_label,time,time_label,value
34,A,Annual,NR,Number,ELC,Electricity,ES,Spain,2013,2013,2021
35,A,Annual,NR,Number,ELC,Electricity,ES,Spain,2014,2014,2832
36,A,Annual,NR,Number,ELC,Electricity,ES,Spain,2015,2015,5243
37,A,Annual,NR,Number,ELC,Electricity,ES,Spain,2016,2016,5756
38,A,Annual,NR,Number,ELC,Electricity,ES,Spain,2017,2017,9681
...,...,...,...,...,...,...,...,...,...,...,...
924,A,Annual,NR,Number,ELC_DIE_PI,Plug-in hybrid diesel-electric,DE,Germany,2020,2020,29846
925,A,Annual,NR,Number,ELC_DIE_PI,Plug-in hybrid diesel-electric,DE,Germany,2021,2021,49822
926,A,Annual,NR,Number,ELC_DIE_PI,Plug-in hybrid diesel-electric,DE,Germany,2022,2022,62162
927,A,Annual,NR,Number,ELC_DIE_PI,Plug-in hybrid diesel-electric,DE,Germany,2023,2023,63918


In [85]:
#looking just at 2024 data for all the countries and sorting it by the value column in descending order to see which country has the most electric vehicles in 2024
total_ev_2024 = df_total_ev[df_total_ev["time"] == "2024"].sort_values(by="value", ascending=False)
total_ev_2024

,freq,freq_label,unit,unit_label,mot_nrg,mot_nrg_label,geo,geo_label,time,time_label,value


In [86]:
#pivot showing total EV per country per year
pivot_df = df_total_ev.pivot_table(index='geo', columns='time', values=['value'], aggfunc='sum')
pivot_df

value                                                           \
time    2013    2014    2015    2016    2017    2018    2019     2020   
geo                                                                     
DE     12000   19000   26000   34022  290491  424487  685801  1312898   
ES      2021   58624   67639  112986  171376  252654  372590   544914   
FR    128409  179947  254132  328493  428637  554140  706376  1049489   
NL      4161    6825  210609  245427  271785  314311  401917   526122   
PT     11356   14109   18597   25000   37436   56639   84452   119247   

                                          
time     2021     2022     2023     2024  
geo                                       
DE    2287209  3350643  4319725  5208437  
ES     817827  1126194  1535170  2024641  
FR    1652083  2279596  3088777  4039369  
NL     724986   957886  1254510  1621141  
PT     171992   239095   343965   472719

In [87]:
#calculating the growth of EVs from 2013 to 2024 for each country by taking the difference between the two years
ev_growth = pivot_df.diff(axis=1)

ev_growth

value                                                                \
time  2013   2014    2015   2016    2017    2018    2019    2020    2021   
geo                                                                        
DE     NaN   7000    7000   8022  256469  133996  261314  627097  974311   
ES     NaN  56603    9015  45347   58390   81278  119936  172324  272913   
FR     NaN  51538   74185  74361  100144  125503  152236  343113  602594   
NL     NaN   2664  203784  34818   26358   42526   87606  124205  198864   
PT     NaN   2753    4488   6403   12436   19203   27813   34795   52745   

                               
time     2022    2023    2024  
geo                            
DE    1063434  969082  888712  
ES     308367  408976  489471  
FR     627513  809181  950592  
NL     232900  296624  366631  
PT      67103  104870  128754

In [88]:
ev_growth_percent = pivot_df.pct_change(axis=1) * 100
ev_growth_percent = ev_growth_percent.round(2)

ev_growth_percent

value                                                               \
time  2013     2014     2015   2016    2017   2018   2019   2020   2021   
geo                                                                       
DE     NaN    58.33    36.84  30.85  753.83  46.13  61.56  91.44  74.21   
ES     NaN  2800.74    15.38  67.04   51.68  47.43  47.47  46.25  50.08   
FR     NaN    40.14    41.23  29.26   30.49  29.28  27.47  48.57  57.42   
NL     NaN    64.02  2985.85  16.53   10.74  15.65  27.87  30.90  37.80   
PT     NaN    24.24    31.81  34.43   49.74  51.30  49.11  41.20  44.23   

                           
time   2022   2023   2024  
geo                        
DE    46.49  28.92  20.57  
ES    37.71  36.31  31.88  
FR    37.98  35.50  30.78  
NL    32.12  30.97  29.23  
PT    39.02  43.86  37.43

In [89]:
#only full electric vehiclesin 2024
df_2024_ev_ele = total_ev_2024[
    total_ev_2024["mot_nrg_label"] == "Electricity"
]
pivot_df_2024_ev = df_2024_ev_ele.pivot_table(index='geo', columns='time', values=['value'], aggfunc='sum')
pivot_df_2024_ev

geo


In [90]:
#opening the data for the total population per 1k cars to be able to calculate the number of EVs per 100k cars for each country
url = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/road_eqs_carhab?lang=en"

response = requests.get(url)
response.raise_for_status()

df_pop = response.json()

df_pop

df_carhab = jsonstat_to_df(df_pop)

df_carhab.sample(5)

,freq,freq_label,unit,unit_label,geo,geo_label,time,time_label,value
142,A,Annual,NR,Number,DK,Denmark,2013,2013,405
70,A,Annual,NR,Number,BG,Bulgaria,2009,2009,337
864,A,Annual,NR,Number,SK,Slovakia,2019,2019,439
16,A,Annual,NR,Number,BE,Belgium,1990,1990,387
1174,A,Annual,NR,Number,AL,Albania,2001,2001,44


In [91]:
df_carhab["geo"].value_counts()

geo
PL           35
CY           35
TR           35
LI           35
SE           35
FI           35
SK           35
SI           35
BE           35
AT           35
NL           35
LT           35
LV           35
HU           35
IT           35
EL           35
BG           35
DK           35
DE           35
HR           35
IE           35
RO           34
FR           34
EE           34
CH           34
MT           33
CZ           33
NO           32
LU           32
ES           31
UK           31
PT           28
MK           23
IS           22
MD           16
EU27_2020    16
AL           15
RS           15
BA           15
GE           13
ME           12
XK           12
Name: count, dtype: int64

In [92]:
#filtering the data to only include the countries we are interested in (PT, NL, DE, FR, ES) and dropping the columns that we don't need for our analysis
df_pop2 = df_carhab[df_carhab["geo"].isin(["PT","NL","DE","FR","ES"])]
df_pop2.sample(15)

,freq,freq_label,unit,unit_label,geo,geo_label,time,time_label,value
667,A,Annual,NR,Number,NL,Netherlands,2024,2024,513
331,A,Annual,NR,Number,FR,France,1997,1997,435
647,A,Annual,NR,Number,NL,Netherlands,2004,2004,429
651,A,Annual,NR,Number,NL,Netherlands,2008,2008,457
741,A,Annual,NR,Number,PT,Portugal,1993,1993,330
751,A,Annual,NR,Number,PT,Portugal,2010,2010,444
171,A,Annual,NR,Number,DE,Germany,2007,2007,501
343,A,Annual,NR,Number,FR,France,2010,2010,538
352,A,Annual,NR,Number,FR,France,2019,2019,569
320,A,Annual,NR,Number,ES,Spain,2021,2021,554


In [93]:
#calculating the number of cars per 100k people for each country by multiplying the value column by 100
df_pop3 = df_pop2.copy()
df_pop3["value_per_100k"] = df_pop3["value"] *100
df_pop3.sample(5)


,freq,freq_label,unit,unit_label,geo,geo_label,time,time_label,value,value_per_100k
654,A,Annual,NR,Number,NL,Netherlands,2011,2011,470,47000
662,A,Annual,NR,Number,NL,Netherlands,2019,2019,493,49300
644,A,Annual,NR,Number,NL,Netherlands,2001,2001,417,41700
325,A,Annual,NR,Number,FR,France,1991,1991,406,40600
169,A,Annual,NR,Number,DE,Germany,2005,2005,559,55900


In [94]:
#changing the time column to an integer to be able to filter the data by year
df_pop4= df_pop3.copy()
df_pop4["time"] = df_pop4["time"].astype(int)
df_pop4.info()

<class 'pandas.core.frame.DataFrame'>
Index: 163 entries, 154 to 765
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   freq            163 non-null    object
 1   freq_label      163 non-null    object
 2   unit            163 non-null    object
 3   unit_label      163 non-null    object
 4   geo             163 non-null    object
 5   geo_label       163 non-null    object
 6   time            163 non-null    int64 
 7   time_label      163 non-null    object
 8   value           163 non-null    int64 
 9   value_per_100k  163 non-null    int64 
dtypes: int64(3), object(7)
memory usage: 14.0+ KB


In [95]:
df_pop4=df_pop4[df_pop4["time"].between(2013, 2024)]
df_pop4["time"].unique()


array([2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023,
       2024])

In [96]:
df_pop4.info()

<class 'pandas.core.frame.DataFrame'>
Index: 60 entries, 177 to 765
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   freq            60 non-null     object
 1   freq_label      60 non-null     object
 2   unit            60 non-null     object
 3   unit_label      60 non-null     object
 4   geo             60 non-null     object
 5   geo_label       60 non-null     object
 6   time            60 non-null     int64 
 7   time_label      60 non-null     object
 8   value           60 non-null     int64 
 9   value_per_100k  60 non-null     int64 
dtypes: int64(3), object(7)
memory usage: 5.2+ KB


In [97]:
df_pop4.sample(5)


,freq,freq_label,unit,unit_label,geo,geo_label,time,time_label,value,value_per_100k
186,A,Annual,NR,Number,DE,Germany,2022,2022,587,58700
657,A,Annual,NR,Number,NL,Netherlands,2014,2014,472,47200
659,A,Annual,NR,Number,NL,Netherlands,2016,2016,481,48100
323,A,Annual,NR,Number,ES,Spain,2024,2024,543,54300
322,A,Annual,NR,Number,ES,Spain,2023,2023,551,55100


In [98]:
pivot_df_pop = df_pop4.pivot_table(index='geo', columns='time', values=['value_per_100k'], aggfunc='sum')
pivot_df_pop

value_per_100k                                                          \
time           2013   2014   2015   2016   2017   2018   2019   2020   2021   
geo                                                                           
DE            54300  54700  54800  55500  56100  56700  57400  58000  58300   
ES            47400  47500  48200  49200  52900  53900  54600  54900  55400   
FR            54600  55100  55800  56400  56900  56800  56900  56800  57000   
NL            47100  47200  47700  48100  48700  48900  49300  49700  50200   
PT            41400  45200  45600  46900  49000  51100  52500  53500  54100   

                           
time   2022   2023   2024  
geo                        
DE    58700  58800  59000  
ES    55300  55100  54300  
FR    56900  57300  57700  
NL    50100  50500  51300  
PT    55200  55000  57100

In [99]:
#filtering the data to only include rows where the "mot_nrg_label" column contains the word "Total"
df_total_cars = df_total[df_total["mot_nrg_label"] == "Total"]
df_total_cars.sample(10)

,freq,freq_label,unit,unit_label,mot_nrg,mot_nrg_label,geo,geo_label,time,time_label,value
8,A,Annual,NR,Number,TOTAL,Total,ES,Spain,2021,2021,26293882
391,A,Annual,NR,Number,TOTAL,Total,FR,France,2015,2015,37180101
7,A,Annual,NR,Number,TOTAL,Total,ES,Spain,2020,2020,26034347
6,A,Annual,NR,Number,TOTAL,Total,ES,Spain,2019,2019,25836586
606,A,Annual,NR,Number,TOTAL,Total,NL,Netherlands,2014,2014,7979083
797,A,Annual,NR,Number,TOTAL,Total,DE,Germany,2013,2013,43851000
3,A,Annual,NR,Number,TOTAL,Total,ES,Spain,2016,2016,22876830
392,A,Annual,NR,Number,TOTAL,Total,FR,France,2016,2016,37673476
178,A,Annual,NR,Number,TOTAL,Total,PT,Portugal,2016,2016,4850229
1,A,Annual,NR,Number,TOTAL,Total,ES,Spain,2014,2014,22029512


In [100]:
#we need it to know hoew many eltric cars per geo to then do the % of the tot
df_total_ev.sample(10)

,freq,freq_label,unit,unit_label,mot_nrg,mot_nrg_label,geo,geo_label,time,time_label,value
43,A,Annual,NR,Number,ELC,Electricity,ES,Spain,2022,2022,95617
664,A,Annual,NR,Number,ELC,Electricity,NL,Netherlands,2018,2018,44581
499,A,Annual,NR,Number,ELC_PET_PI,Plug-in hybrid petrol-electric,FR,France,2015,2015,35766
291,A,Annual,NR,Number,ELC_PET_PI,Plug-in hybrid petrol-electric,PT,Portugal,2022,2022,56740
108,A,Annual,NR,Number,ELC_DIE_HYB,Hybrid diesel-electric,ES,Spain,2021,2021,79462
325,A,Annual,NR,Number,ELC_DIE_PI,Plug-in hybrid diesel-electric,PT,Portugal,2021,2021,7268
272,A,Annual,NR,Number,ELC_PET_HYB,Hybrid electric-petrol,PT,Portugal,2014,2014,11428
240,A,Annual,NR,Number,ELC,Electricity,PT,Portugal,2018,2018,9980
669,A,Annual,NR,Number,ELC,Electricity,NL,Netherlands,2023,2023,444669
68,A,Annual,NR,Number,ELC_PET_HYB,Hybrid electric-petrol,ES,Spain,2014,2014,54362


In [101]:
df_total_ev2= df_total_ev.copy()

In [102]:
# Group the data by geographical location,  and time, then sum the values fo all types of electric vehicles to get the total number of electric vehicles per country per year
df_total_ev_grouped = (
    df_total_ev
    .groupby(["geo_label", "time"], as_index=False)["value"]
    .sum()
)
df_total_ev_grouped.head()

,geo_label,time,value
0,France,2013,128409
1,France,2014,179947
2,France,2015,254132
3,France,2016,328493
4,France,2017,428637


In [103]:
#we double checked if germany was ok
df_total_germany = df_total_ev[df_total_ev["geo_label"] == "Germany"]
df_total_germany = df_total_germany[df_total_germany["time"] == 2017]
df_total_germany

,freq,freq_label,unit,unit_label,mot_nrg,mot_nrg_label,geo,geo_label,time,time_label,value
861,A,Annual,NR,Number,ELC,Electricity,DE,Germany,2017,2017,53861
889,A,Annual,NR,Number,ELC_PET_HYB,Hybrid electric-petrol,DE,Germany,2017,2017,181885
897,A,Annual,NR,Number,ELC_PET_PI,Plug-in hybrid petrol-electric,DE,Germany,2017,2017,42569
913,A,Annual,NR,Number,ELC_DIE_HYB,Hybrid diesel-electric,DE,Germany,2017,2017,10336
921,A,Annual,NR,Number,ELC_DIE_PI,Plug-in hybrid diesel-electric,DE,Germany,2017,2017,1840


In [104]:
df_total_ev_grouped = df_total_ev_grouped.rename(
    columns={"value": "total_ev_value"}
)

df_total_cars_clean = df_total_cars.rename(
    columns={"value": "total_cars_value"}
)

In [105]:
#merging the two dataframes together to have a complete dataset for all the countries with the total number of electric vehicles and the total number of cars per country per year
df_combined = df_total_ev_grouped.merge(
    df_total_cars_clean,
    on=["geo_label", "time"],
    how="inner"
)

df_combined.head()

,geo_label,time,total_ev_value,freq,freq_label,unit,unit_label,mot_nrg,mot_nrg_label,geo,time_label,total_cars_value
0,France,2013,128409,A,Annual,NR,Number,TOTAL,Total,FR,2013,36149230
1,France,2014,179947,A,Annual,NR,Number,TOTAL,Total,FR,2014,36647883
2,France,2015,254132,A,Annual,NR,Number,TOTAL,Total,FR,2015,37180101
3,France,2016,328493,A,Annual,NR,Number,TOTAL,Total,FR,2016,37673476
4,France,2017,428637,A,Annual,NR,Number,TOTAL,Total,FR,2017,38117663


In [106]:
#calculating the percentage of electric vehicles per total cars for each country per year by dividing the total number of electric vehicles by the total number of cars and multiplying by 100 to get the percentage
df_combined["ev_per_total_cars"] = (df_combined["total_ev_value"] / df_combined["total_cars_value"]) * 100
df_combined.sample(15)

,geo_label,time,total_ev_value,freq,freq_label,unit,unit_label,mot_nrg,mot_nrg_label,geo,time_label,total_cars_value,ev_per_total_cars
1,France,2014,179947,A,Annual,NR,Number,TOTAL,Total,FR,2014,36647883,0.491016
50,Spain,2015,67639,A,Annual,NR,Number,TOTAL,Total,ES,2015,22355549,0.302560
29,Netherlands,2018,314311,A,Annual,NR,Number,TOTAL,Total,NL,2018,8442982,3.722749
10,France,2023,3088777,A,Annual,NR,Number,TOTAL,Total,FR,2023,39358421,7.847817
30,Netherlands,2019,401917,A,Annual,NR,Number,TOTAL,Total,NL,2019,8584391,4.681951
25,Netherlands,2014,6825,A,Annual,NR,Number,TOTAL,Total,NL,2014,7979083,0.085536
9,France,2022,2279596,A,Annual,NR,Number,TOTAL,Total,FR,2022,38973339,5.849116
42,Portugal,2019,84452,A,Annual,NR,Number,TOTAL,Total,PT,2019,5452119,1.548976
57,Spain,2022,1126194,A,Annual,NR,Number,TOTAL,Total,ES,2022,26605478,4.232940
58,Spain,2023,1535170,A,Annual,NR,Number,TOTAL,Total,ES,2023,26778142,5.732922


In [107]:
drop_list=["time_label","unit_label","unit","freq","freq_label","mot_nrg_label"]
df_combined_clean = df_combined.drop(columns=drop_list)
df_combined_clean.sample(10)

,geo_label,time,total_ev_value,mot_nrg,geo,total_cars_value,ev_per_total_cars
4,France,2017,428637,TOTAL,FR,38117663,1.124510
56,Spain,2021,817827,TOTAL,ES,26293882,3.110332
12,Germany,2013,12000,TOTAL,DE,43851000,0.027365
29,Netherlands,2018,314311,TOTAL,NL,8442982,3.722749
54,Spain,2019,372590,TOTAL,ES,25836586,1.442102
11,France,2024,4039369,TOTAL,FR,39738933,10.164765
59,Spain,2024,2024641,TOTAL,ES,26700972,7.582649
38,Portugal,2015,18597,TOTAL,PT,4722963,0.393757
1,France,2014,179947,TOTAL,FR,36647883,0.491016
32,Netherlands,2021,724986,TOTAL,NL,8827709,8.212618


In [108]:
df_pop4.head()


,freq,freq_label,unit,unit_label,geo,geo_label,time,time_label,value,value_per_100k
177,A,Annual,NR,Number,DE,Germany,2013,2013,543,54300
178,A,Annual,NR,Number,DE,Germany,2014,2014,547,54700
179,A,Annual,NR,Number,DE,Germany,2015,2015,548,54800
180,A,Annual,NR,Number,DE,Germany,2016,2016,555,55500
181,A,Annual,NR,Number,DE,Germany,2017,2017,561,56100


In [109]:
# Combine the cleaned data with population data
df_combined_all = df_combined_clean.merge(
    df_pop4,
    on=["geo", "geo_label", "time"],
    how="inner"
)
df_combined_all.sample(10)

,geo_label,time,total_ev_value,mot_nrg,geo,total_cars_value,ev_per_total_cars,freq,freq_label,unit,unit_label,time_label,value,value_per_100k
13,Germany,2014,19000,TOTAL,DE,44403000,0.042790,A,Annual,NR,Number,2014,547,54700
43,Portugal,2020,119247,TOTAL,PT,5565963,2.142432,A,Annual,NR,Number,2020,535,53500
1,France,2014,179947,TOTAL,FR,36647883,0.491016,A,Annual,NR,Number,2014,551,55100
35,Netherlands,2024,1621141,TOTAL,NL,9247810,17.529999,A,Annual,NR,Number,2024,513,51300
15,Germany,2016,34022,TOTAL,DE,45803560,0.074278,A,Annual,NR,Number,2016,555,55500
6,France,2019,706376,TOTAL,FR,38423556,1.838393,A,Annual,NR,Number,2019,569,56900
30,Netherlands,2019,401917,TOTAL,NL,8584391,4.681951,A,Annual,NR,Number,2019,493,49300
28,Netherlands,2017,271785,TOTAL,NL,8373244,3.245875,A,Annual,NR,Number,2017,487,48700
54,Spain,2019,372590,TOTAL,ES,25836586,1.442102,A,Annual,NR,Number,2019,546,54600
12,Germany,2013,12000,TOTAL,DE,43851000,0.027365,A,Annual,NR,Number,2013,543,54300


In [110]:
df_combined_all= df_combined_all.rename(columns={"value": "cars_per_1k"})
df_combined_all= df_combined_all.rename(columns={"value_per_100k": "cars_per_100k"})

In [111]:
# Pivot the data for Spain
df_spain_pivot = (
    df_combined_all[df_combined_all["geo"] == "ES"]
    .pivot_table(
        index="time",
        values=["total_ev_value", "total_cars_value","geo", "cars_per_100k","ev_per_total_cars"],
        aggfunc="sum"
    )
    .reset_index()
)

df_spain_pivot

,time,cars_per_100k,ev_per_total_cars,geo,total_cars_value,total_ev_value
0,2013,47400,0.009176,ES,22025000,2021
1,2014,47500,0.266116,ES,22029512,58624
2,2015,48200,0.302560,ES,22355549,67639
3,2016,49200,0.493888,ES,22876830,112986
4,2017,52900,0.694489,ES,24676557,171376
5,2018,53900,0.998467,ES,25304190,252654
6,2019,54600,1.442102,ES,25836586,372590
7,2020,54900,2.093058,ES,26034347,544914
8,2021,55400,3.110332,ES,26293882,817827
9,2022,55300,4.232940,ES,26605478,1126194


In [112]:
df_combined_all.head()

,geo_label,time,total_ev_value,mot_nrg,geo,total_cars_value,ev_per_total_cars,freq,freq_label,unit,unit_label,time_label,cars_per_1k,cars_per_100k
0,France,2013,128409,TOTAL,FR,36149230,0.355219,A,Annual,NR,Number,2013,546,54600
1,France,2014,179947,TOTAL,FR,36647883,0.491016,A,Annual,NR,Number,2014,551,55100
2,France,2015,254132,TOTAL,FR,37180101,0.683516,A,Annual,NR,Number,2015,558,55800
3,France,2016,328493,TOTAL,FR,37673476,0.871948,A,Annual,NR,Number,2016,564,56400
4,France,2017,428637,TOTAL,FR,38117663,1.124510,A,Annual,NR,Number,2017,569,56900


In [113]:
df_combined_all["ev_per_100k"] = (
    df_combined_all["cars_per_100k"] * df_combined_all["ev_per_total_cars"] / 100
).round(0).astype(int)

df_combined_all.head()

,geo_label,time,total_ev_value,mot_nrg,geo,total_cars_value,ev_per_total_cars,freq,freq_label,unit,unit_label,time_label,cars_per_1k,cars_per_100k,ev_per_100k
0,France,2013,128409,TOTAL,FR,36149230,0.355219,A,Annual,NR,Number,2013,546,54600,194
1,France,2014,179947,TOTAL,FR,36647883,0.491016,A,Annual,NR,Number,2014,551,55100,271
2,France,2015,254132,TOTAL,FR,37180101,0.683516,A,Annual,NR,Number,2015,558,55800,381
3,France,2016,328493,TOTAL,FR,37673476,0.871948,A,Annual,NR,Number,2016,564,56400,492
4,France,2017,428637,TOTAL,FR,38117663,1.124510,A,Annual,NR,Number,2017,569,56900,640


In [114]:
df_combined_all= df_combined_all.rename(columns={"total_cars_value": "total_cars"})
df_combined_all= df_combined_all.rename(columns={"total_ev_value": "total_ev"})
df_combined_all.head()

,geo_label,time,total_ev,mot_nrg,geo,total_cars,ev_per_total_cars,freq,freq_label,unit,unit_label,time_label,cars_per_1k,cars_per_100k,ev_per_100k
0,France,2013,128409,TOTAL,FR,36149230,0.355219,A,Annual,NR,Number,2013,546,54600,194
1,France,2014,179947,TOTAL,FR,36647883,0.491016,A,Annual,NR,Number,2014,551,55100,271
2,France,2015,254132,TOTAL,FR,37180101,0.683516,A,Annual,NR,Number,2015,558,55800,381
3,France,2016,328493,TOTAL,FR,37673476,0.871948,A,Annual,NR,Number,2016,564,56400,492
4,France,2017,428637,TOTAL,FR,38117663,1.124510,A,Annual,NR,Number,2017,569,56900,640


In [115]:
df_spain_pivot = (
    df_combined_all[df_combined_all["geo"] == "ES"]
    .pivot_table(
        index="time",
        values=["total_ev", "total_cars","geo", "cars_per_100k","ev_per_total_cars","ev_per_100k"],
        aggfunc="sum"
    )
    .reset_index()
)

df_spain_pivot

,time,cars_per_100k,ev_per_100k,ev_per_total_cars,geo,total_cars,total_ev
0,2013,47400,4,0.009176,ES,22025000,2021
1,2014,47500,126,0.266116,ES,22029512,58624
2,2015,48200,146,0.302560,ES,22355549,67639
3,2016,49200,243,0.493888,ES,22876830,112986
4,2017,52900,367,0.694489,ES,24676557,171376
5,2018,53900,538,0.998467,ES,25304190,252654
6,2019,54600,787,1.442102,ES,25836586,372590
7,2020,54900,1149,2.093058,ES,26034347,544914
8,2021,55400,1723,3.110332,ES,26293882,817827
9,2022,55300,2341,4.232940,ES,26605478,1126194


In [116]:
df_all_geo_grouped = (
    df_combined_all
    .groupby(["geo_label", "time"], as_index=False)[["ev_per_100k", "total_ev", "total_cars"]]
    .sum()
)
df_all_geo_grouped

,geo_label,time,ev_per_100k,total_ev,total_cars
0,France,2013,194,128409,36149230
1,France,2014,271,179947,36647883
2,France,2015,381,254132,37180101
3,France,2016,492,328493,37673476
4,France,2017,640,428637,38117663
5,France,2018,823,554140,38229305
6,France,2019,1046,706376,38423556
7,France,2020,1550,1049489,38470122
8,France,2021,2426,1652083,38818914
9,France,2022,3328,2279596,38973339


In [117]:
# Calculate the growth rate of ev_per_100k for each geo_label
df_all_geo_grouped["ev_per_100k_growth_rate"] = (
    df_all_geo_grouped
    .groupby("geo_label")["ev_per_100k"]
    .pct_change() * 100
)

In [118]:
df_all_geo_grouped

,geo_label,time,ev_per_100k,total_ev,total_cars,ev_per_100k_growth_rate
0,France,2013,194,128409,36149230,NaN
1,France,2014,271,179947,36647883,39.690722
2,France,2015,381,254132,37180101,40.590406
3,France,2016,492,328493,37673476,29.133858
4,France,2017,640,428637,38117663,30.081301
5,France,2018,823,554140,38229305,28.593750
6,France,2019,1046,706376,38423556,27.095990
7,France,2020,1550,1049489,38470122,48.183556
8,France,2021,2426,1652083,38818914,56.516129
9,France,2022,3328,2279596,38973339,37.180544


In [119]:
df_all_geo_grouped = df_all_geo_grouped[
    df_all_geo_grouped["time"].astype(int) >= 2017
]

In [120]:
df_all_geo_grouped["ev_per_100k_growth_rate"] = df_all_geo_grouped["ev_per_100k_growth_rate"].round(2)

C:\Users\jejoc\AppData\Local\Temp\ipykernel_27340\3315554817.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_all_geo_grouped["ev_per_100k_growth_rate"] = df_all_geo_grouped["ev_per_100k_growth_rate"].round(2)


In [121]:
df_all_geo_grouped

,geo_label,time,ev_per_100k,total_ev,total_cars,ev_per_100k_growth_rate
4,France,2017,640,428637,38117663,30.08
5,France,2018,823,554140,38229305,28.59
6,France,2019,1046,706376,38423556,27.10
7,France,2020,1550,1049489,38470122,48.18
8,France,2021,2426,1652083,38818914,56.52
9,France,2022,3328,2279596,38973339,37.18
10,France,2023,4497,3088777,39358421,35.13
11,France,2024,5865,4039369,39738933,30.42
16,Germany,2017,351,290491,46474594,756.10
17,Germany,2018,511,424487,47095784,45.58


In [122]:
## WEBSCARPING

In [123]:
# 1- Pick the URL and save it
news_url = "https://www.globalbankingandfinance.com/electriccars-spain-one/"

# 2- Identify ourselves as a browser (tested and working!) = HEADERS
headers = {"User-Agent":"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/116.0.0.0 Safari/537.36"}
# "I am Chrome 116, running on Windows 10 64-bit, using WebKit as my rendering engine."

# 3- Make the request, passing the URL and the HEADERS
news_response = requests.get(news_url, headers=headers)

# 4- Check the status (Good Practice!)
if news_response.status_code == 200:
    print("Connection successful!")
else:
    print(f"Connection failed: {news_response.status_code}")

Connection successful!


In [124]:
#printing the status code and first 1000 characters of the articile
print(news_response.status_code)
print(news_response.text[:1000])

200
<!DOCTYPE html><html lang="en"> <head><meta charset="UTF-8"><meta name="viewport" content="width=device-width, initial-scale=1, minimum-scale=1, maximum-scale=6, viewport-fit=cover"><link rel="preload" as="image" href="https://cdn.sanity.io/images/v0gkry1w/production/e1d03ab7c40a2eeba422bc1806547c8587db3e3b-640x430.webp?w=720&#38;q=72&#38;fm=webp&#38;fit=crop&#38;auto=format" imagesrcset="https://cdn.sanity.io/images/v0gkry1w/production/e1d03ab7c40a2eeba422bc1806547c8587db3e3b-640x430.webp?w=400&#38;q=72&#38;fm=webp&#38;fit=crop&#38;auto=format 400w, https://cdn.sanity.io/images/v0gkry1w/production/e1d03ab7c40a2eeba422bc1806547c8587db3e3b-640x430.webp?w=640&#38;q=72&#38;fm=webp&#38;fit=crop&#38;auto=format 640w, https://cdn.sanity.io/images/v0gkry1w/production/e1d03ab7c40a2eeba422bc1806547c8587db3e3b-640x430.webp?w=720&#38;q=72&#38;fm=webp&#38;fit=crop&#38;auto=format 720w, https://cdn.sanity.io/images/v0gkry1w/production/e1d03ab7c40a2eeba422bc1806547c8587db3e3b-640x430.webp?w=1024

In [125]:
from bs4 import BeautifulSoup

In [126]:
#Parcing the response
news_soup = BeautifulSoup(news_response.content, "html.parser")
print(news_soup.prettify()[:2000])

<!DOCTYPE html>
<html lang="en">
 <head>
  <meta charset="utf-8"/>
  <meta content="width=device-width, initial-scale=1, minimum-scale=1, maximum-scale=6, viewport-fit=cover" name="viewport"/>
  <link as="image" fetchpriority="high" href="https://cdn.sanity.io/images/v0gkry1w/production/e1d03ab7c40a2eeba422bc1806547c8587db3e3b-640x430.webp?w=720&amp;q=72&amp;fm=webp&amp;fit=crop&amp;auto=format" imagesizes="(max-width: 768px) 100vw, (max-width: 1024px) 90vw, (max-width: 1536px) 720px, 660px" imagesrcset="https://cdn.sanity.io/images/v0gkry1w/production/e1d03ab7c40a2eeba422bc1806547c8587db3e3b-640x430.webp?w=400&amp;q=72&amp;fm=webp&amp;fit=crop&amp;auto=format 400w, https://cdn.sanity.io/images/v0gkry1w/production/e1d03ab7c40a2eeba422bc1806547c8587db3e3b-640x430.webp?w=640&amp;q=72&amp;fm=webp&amp;fit=crop&amp;auto=format 640w, https://cdn.sanity.io/images/v0gkry1w/production/e1d03ab7c40a2eeba422bc1806547c8587db3e3b-640x430.webp?w=720&amp;q=72&amp;fm=webp&amp;fit=crop&amp;auto=format 7

In [127]:
import requests
from bs4 import BeautifulSoup

# 1 - Pick the URL
news_url = "https://www.globalbankingandfinance.com/electriccars-spain-one/"

# 2 - Identify ourselves as a browser
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/116.0.0.0 Safari/537.36"
}

# 3 - Make the request
news_response = requests.get(news_url, headers=headers)

# 4 - Check if request worked
print(news_response.status_code)

# 5 - Parse the HTML
news_soup = BeautifulSoup(news_response.content, "html.parser")

# 6 - Extract page title
print(news_soup.title.text)

200
Spain's $1.5 Billion EV Boost - Electric Vehicle Market Growth | Global Banking & Finance Review


In [128]:
# A quick shortcut: soup.title gives us the <title> tag directly
print(news_soup.title)                   # Full tag: <title>Spain's $1.5 Billion EV Boost - Electric Vehicle Market Growth | Global Banking &amp; Finance Review</title>
print(news_soup.title.string)            # Just the text: Spain's $1.5 Billion EV Boost - Electric Vehicle Market Growth | Global Banking & Finance Review

<title>Spain's $1.5 Billion EV Boost - Electric Vehicle Market Growth | Global Banking &amp; Finance Review</title>
Spain's $1.5 Billion EV Boost - Electric Vehicle Market Growth | Global Banking & Finance Review


In [129]:
news_para = news_soup.find_all("p")
print(len(news_para))

23


In [130]:
news_para[0:13]

[<p class="px-4 py-2 border border-gray-200 mt-2 text-sm leading-relaxed text-gray-700" data-astro-cid-qusqgt37=""> Global Banking &amp; Finance Review® is an online platform offering news, analysis, and opinion on the latest trends, developments, and innovations in the banking and finance industry worldwide. The platform covers a diverse range of topics, including banking, insurance, investment, wealth management, fintech, and regulatory issues. The website publishes news, press releases, opinion and advertorials on various financial organizations, products and services which are commissioned from various Companies, Organizations, PR agencies, Bloggers etc. These commissioned articles are commercial in nature. This is not to be considered as financial advice and should be considered only for information purposes. It does not reflect the views or opinion of our website and is not to be considered an endorsement or a recommendation. We cannot guarantee the accuracy or applicability of a

In [131]:
article_paragraphs = news_para[5:]
print(article_paragraphs)

[<p>MADRID, Dec 3 (Reuters) - Spain will provide nearly 1.3 billion euros ($1.52 billion) to support its electric vehicle market and industry next year as part of a plan to lift the share of EVs produced in the country to 95% by 2035, Prime Minister Pedro Sanchez said on Wednesday.</p>, <p>In the first 10 months of 2025, the share of fully electric and plug-in hybrid vehicles made in Spain totalled around 10%, industry data show. Self-charging hybrids accounted for 26.7%.</p>, <p>Around 20% of vehicles across the EU last year were fully electric or plug-in hybrids.</p>, <p>Spain's plan includes 400 million euros in direct subsidies in 2026 for consumers to buy EVs and another 580 million euros under the country's EU-funded scheme supporting industrial investment.</p>, <p>It will also add 300 million euros to install charging points along roads still lacking coverage.</p>, <p>Spain is stepping up support for its automotive sector as Chinese EV brands like BYD rapidly expand, undercuttin

In [132]:
article_text_list = []

for p in article_paragraphs:
    text = p.get_text(strip=True)
    article_text_list.append(text)
    
print(article_text_list)

['MADRID, Dec 3 (Reuters) - Spain will provide nearly 1.3 billion euros ($1.52 billion) to support its electric vehicle market and industry next year as part of a plan to lift the share of EVs produced in the country to 95% by 2035, Prime Minister Pedro Sanchez said on Wednesday.', 'In the first 10 months of 2025, the share of fully electric and plug-in hybrid vehicles made in Spain totalled around 10%, industry data show. Self-charging hybrids accounted for 26.7%.', 'Around 20% of vehicles across the EU last year were fully electric or plug-in hybrids.', "Spain's plan includes 400 million euros in direct subsidies in 2026 for consumers to buy EVs and another 580 million euros under the country's EU-funded scheme supporting industrial investment.", 'It will also add 300 million euros to install charging points along roads still lacking coverage.', "Spain is stepping up support for its automotive sector as Chinese EV brands like BYD rapidly expand, undercutting European rivals and explo

In [133]:
article_text = "\n\n".join(article_text_list)

In [134]:
print(article_text)

MADRID, Dec 3 (Reuters) - Spain will provide nearly 1.3 billion euros ($1.52 billion) to support its electric vehicle market and industry next year as part of a plan to lift the share of EVs produced in the country to 95% by 2035, Prime Minister Pedro Sanchez said on Wednesday.

In the first 10 months of 2025, the share of fully electric and plug-in hybrid vehicles made in Spain totalled around 10%, industry data show. Self-charging hybrids accounted for 26.7%.

Around 20% of vehicles across the EU last year were fully electric or plug-in hybrids.

Spain's plan includes 400 million euros in direct subsidies in 2026 for consumers to buy EVs and another 580 million euros under the country's EU-funded scheme supporting industrial investment.

It will also add 300 million euros to install charging points along roads still lacking coverage.

Spain is stepping up support for its automotive sector as Chinese EV brands like BYD rapidly expand, undercutting European rivals and exploiting the co

In [135]:
with open("spain_ev_news_article.txt", "w", encoding="utf-8") as file:
    file.write(article_text)